In [ ]:
import pandas as pd

# -------------------------------
# Load and process the game CSV
# -------------------------------



# Load the CSV files
df_hitting = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\USDHITTINGYTD.csv")
df_pitching = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\USDPITCHINGYTD.csv")

# Append the datasets together
df = pd.concat([df_hitting, df_pitching], ignore_index=True)

# Define mapping for pitch types
pitch_mapping = {
    'FA': 'Fast',
    'CU': 'Break',
    'CH': 'Slow',
    'SL': 'Break',
    'SI': 'Fast',
    'FC': 'Fast',
    'UN': None,   # filter out
    'IN': None,   # filter out
    'FF': 'Fast',
    'FS': 'Slow',
    'KN': 'Slow'
}

# Filter out rows where pitchType is one of the ones to filter (i.e. mapping to None)
df = df[df['pitchType'].isin([pt for pt, group in pitch_mapping.items() if group is not None])].copy()

# Map the pitch types to a new column 'pitchgroup'
df['pitchgroup'] = df['pitchType'].map(pitch_mapping)


In [ ]:

# -------------------------------
# Categorize the pitchResult into events
# -------------------------------

def categorize_event(event):
    """
    Categorize the pitchResult into a standardized event label.
    Bunt events are ignored (return None).
    """
    event = event.lower()
    if "bunt" in event or "Unknown" in event:
        return None
    elif "single" in event:
        return "single"
    elif "double play" in event:
        return "field_out"
    elif "double" in event:
        return "double"
    elif "triple" in event:
        return "triple"
    elif "home run" in event:
        return "home_run"
    elif "looking" in event:
        return "called_strike"
    elif "swinging" in event:
        return "swinging_strike"
    elif "hit by pitch" in event:
        return "hit_by_pitch"
    elif "walk" in event or "ball" in event:
        return "ball"
    elif "foul" in event:
        return "foul"
    elif ("line out" in event or "fly out" in event or "ground out" in event or 
          "pop out" in event or "double play" in event or "reached on error" in event or 
          "in play out" in event or "sac fly" in event or "fielder's choice" in event):
        return "field_out"
    elif "ball in the dirt" in event:
        return "ball"
    else:
        return "unknown"

# Apply the event categorization to the pitchResult column
df['event_category'] = df['pitchResult'].apply(categorize_event)

# Drop rows where event_category is None (e.g. bunts)
df = df[(df['event_category'].notna()) & (df['event_category'] != 'unknown')].copy()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 1. Split 'count' column into 'balls' and 'strikes'
# -------------------------------
df[['balls', 'strikes']] = df['count'].str.split('-', expand=True)
df['balls'] = pd.to_numeric(df['balls'], errors='coerce')
df['strikes'] = pd.to_numeric(df['strikes'], errors='coerce')

# -------------------------------
# 2. Load run_values and merge with df
# -------------------------------
run_values = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\run_values.csv")
run_values = run_values.rename(columns={"event": "event_category"})
df_joined = pd.merge(
    df,
    run_values,
    on=["balls", "strikes", "event_category"],
    how="left"
)

# Identify unique combinations with no join available
no_join_rows = df_joined[df_joined['delta_run_exp'].isnull()][['event_category', 'balls', 'strikes']].drop_duplicates()
print("\nUnique combinations with no available join:")
print(no_join_rows)

# -------------------------------
# 3. Create Binary 'win' Column
# -------------------------------
win_events = {'foul', 'called_strike', 'swinging_strike', 'field_out', 'strikeout'}
df_joined['win'] = df_joined['event_category'].apply(lambda x: 1 if x in win_events else 0)

# -------------------------------
# 4. Adjust PX Orientation
# -------------------------------
df_joined['PX'] = df_joined['PX'] * -1

# -------------------------------
# 5. Create Binary Count Categories
# -------------------------------
df_joined['count_0_0'] = ((df_joined['balls'] == 0) & (df_joined['strikes'] == 0)).astype(int)
df_joined['count_hitters'] = df_joined[['balls', 'strikes']].apply(lambda x: 1 if (x['balls'], x['strikes']) in [(1,0), (2,0), (3,0), (3,1)] else 0, axis=1)
df_joined['count_pitchers'] = df_joined[['balls', 'strikes']].apply(lambda x: 1 if (x['balls'], x['strikes']) in [(0,2), (0,1), (1,2)] else 0, axis=1)
df_joined['count_2k'] = ((df_joined['strikes'] == 2) & (df_joined['balls'] != 3)).astype(int)

# -------------------------------
# 6. Define Additional Binary Features
# -------------------------------
# 6.1 Define Strike (if event_category is one of these)
strike_events = {"foul", "called_strike", "swinging_strike", "field_out", "strikeout",
                 "home_run", "triple", "double", "single"}
df_joined['Strike'] = df_joined['event_category'].isin(strike_events)

# 6.2 Define Comploc (using PX and PZ as per your specification)
df_joined['Comploc'] = df_joined.apply(lambda row: -1.15 <= row['PX'] <= 1.15 and 1.1 <= row['PZ'] <= 3.9, axis=1)

# 6.3 Define Inzone (using PX and PZ)
df_joined['Inzone'] = df_joined.apply(lambda row: -0.83 <= row['PX'] <= 0.83 and 1.5 <= row['PZ'] <= 3.5, axis=1)

# 6.4 Define Swing (if event_category is one of these)
swing_events = {"foul", "swinging_strike", "field_out", "home_run", "triple", "double", "single"}
df_joined['Swing'] = df_joined['event_category'].isin(swing_events)

# -------------------------------
# 7. Define Whiff (as swinging strike)
# -------------------------------
df_joined['Whiff'] = df_joined['pitchResult'].str.lower().str.contains("swinging", na=False).astype(int)


# -------------------------------
# 8. Create delta_run_exp_squared column (custom transformation)
# -------------------------------
df_joined['delta_run_exp_squared'] = df_joined['delta_run_exp'].apply(lambda x: 
    0.5 + (x - 0.5) * 0.5 if x > 0.5 else
    -0.5 + (x + 0.5) * 0.5 if x < -0.5 else
    0.2 + (x - 0.2) * 0.75 if x > 0.2 else
    -0.2 + (x + 0.2) * 0.75 if x < -0.2 else
    x * 2.5)

# -------------------------------
# 9. Convert Numeric Columns
# -------------------------------
numeric_cols = ["Vel", "delta_run_exp", "Extension", "HorzApprAngle", "VertApprAngle", 
                "IndVertBrk", "HorzBrk", "RelZ", "RelX"]
for col in numeric_cols:
    df_joined[col] = pd.to_numeric(df_joined[col], errors='coerce')

# -------------------------------
# 10. Calculate Runs Scored
# -------------------------------
df_joined['Runs Scored'] = np.maximum(0, np.maximum(
    df_joined['opponentCurrentRuns'].shift(-1) - df_joined['opponentCurrentRuns'],
    df_joined['currentRuns'].shift(-1) - df_joined['currentRuns'].fillna(0)
))


# -------------------------------
# 11. Convert gameDate to datetime
# -------------------------------
df_joined['gameDate'] = pd.to_datetime(df_joined['gameDate'])

df_joined['clean_pitchResult'] = df_joined['pitchResult'].str.split(' on a').str[0].str.strip()

event_abbreviations = {
    "Single": "1B",
    "Foul": "Foul",
    "Hit By Pitch": "HBP",
    "Strike Swinging": "SS",
    "Strikeout (Swinging)": "K",  # Forward K for swinging strikeout
    "Strikeout (Looking)": "ꓘ",  # Backward K for looking strikeout
    "Ball": "B",
    "Walk": "BB",
    "Home Run": "HR",
    "Double": "2B",
    "Fielder's Choice": "FC",
    "Triple": "3B",
    "Reached on Error": "ROE",
    "Sac Fly": "SF"
}

# Apply abbreviation mapping
df_joined['clean_pitchResult'] = df_joined['clean_pitchResult'].map(event_abbreviations).fillna(df_joined['clean_pitchResult'])


# -------------------------------
# 12. Create Event_Desc Column
# -------------------------------
df_joined['Event_Desc'] = df_joined.apply(lambda row: (
    f"{row['balls']}-{row['strikes']} " +  # Move balls-strikes to the start
    f"{row['pitchTypeFull']}, " +          # Followed by pitchTypeFull
    (f"{int(row['Runs Scored'])} Run " if row['Runs Scored'] > 0 else '') +
    f"{row['clean_pitchResult']}, " +
    ("Bases Empty" if not (row['ManOn1st'] == 1 or row['ManOn2nd'] == 1 or row['ManOn3rd'] == 1) else "Runners on " +
    " ".join(filter(None, [
        "1st" if row['ManOn1st'] == 1 else '',
        "2nd" if row['ManOn2nd'] == 1 else '',
        "3rd" if row['ManOn3rd'] == 1 else ''
    ]))) +
    f", {row['inn']} " +
    f"{row['outs']} Out"
), axis=1)




# Define events that count as reaching base
valid_leadoff_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}

# Step 1: Identify leadoff batters (first PA in each inning)
df_joined["inning_leadoff"] = df_joined.groupby(["gameDate", "inn"])["abNumInGame"].transform("min") == df_joined["abNumInGame"]

# Step 2: Identify successful leadoff batters (reached base via valid event)
df_joined["inning_leadoff"] = df_joined["inning_leadoff"] & df_joined["event_category"].isin(valid_leadoff_events)

# Step 3: If a leadoff batter succeeded in an inning, mark success for all rows in that inning
df_joined["inning_leadoff_success"] = df_joined.groupby(["gameDate", "inn"])["inning_leadoff"].transform("max")

# Convert to integer (True/False → 1/0)
df_joined["inning_leadoff"] = df_joined["inning_leadoff"].astype(int)
df_joined["inning_leadoff_success"] = df_joined["inning_leadoff_success"].astype(int)
# -------------------------------
# End of Feature Engineering Script
# -------------------------------
print("Feature engineering complete. Here's a preview:")
df_joined.head()


In [ ]:
print(df_joined['clean_pitchResult'].unique())

In [ ]:
win_events = {'foul', 'called_strike', 'swinging_strike', 'field_out', 'strikeout'}

# Create a binary 'win' column (1 if event is a win, 0 otherwise)
df_joined['win'] = df_joined['event_category'].apply(lambda x: 1 if x in win_events else 0)


# Multiply PX by -1 to match standard orientation
df_joined['PX'] = df_joined['PX'] * -1

# Create binary fields for different count categories
df_joined['count_0_0'] = ((df_joined['balls'] == 0) & (df_joined['strikes'] == 0)).astype(int)
df_joined['count_hitters'] = df_joined[['balls', 'strikes']].apply(lambda x: 1 if (x['balls'], x['strikes']) in [(1,0), (2,0), (3,0), (3,1)] else 0, axis=1)
df_joined['count_pitchers'] = df_joined[['balls', 'strikes']].apply(lambda x: 1 if (x['balls'], x['strikes']) in [(0,2), (0,1), (1,2)] else 0, axis=1)
df_joined['count_2k'] = ((df_joined['strikes'] == 2) & (df_joined['balls'] != 3)).astype(int)

# 1) Define Strike
# event_category ∈ {foul, called_strike, swinging_strike, field_out,
#                   strikeout, home_run, triple, double, single}
strike_events = {
    "foul", "called_strike", "swinging_strike", "field_out", "strikeout",
    "home_run", "triple", "double", "single"
}
df_joined['Strike'] = df_joined['event_category'].isin(strike_events)

# 2) Define Comploc
# (PX between -1.15 and 1.15) AND (PZ between 1.1 and 3.9)
df_joined['Comploc'] = df_joined.apply(
    lambda row: -1.15 <= row['PX'] <= 1.15 and 1.1 <= row['PZ'] <= 3.9,
    axis=1
)

# 3) Define Inzone (PlateLocSide & PlateLocHeight)
# (Platelocside between -0.83 and 0.83) AND (Platelocheight between 1.5 and 3.5)
df_joined['Inzone'] = df_joined.apply(
    lambda row: -0.83 <= row['PX'] <= 0.83 and 1.5 <= row['PZ'] <= 3.5,
    axis=1
)

# 4) Define Swing
# event_category ∈ {foul, swinging_strike, field_out, home_run, triple, double, single}
swing_events = {
    "foul", "swinging_strike", "field_out",
    "home_run", "triple", "double", "single"
}
df_joined['Swing'] = df_joined['event_category'].isin(swing_events)

# Quick check
df_joined[['Strike', 'Comploc', 'Inzone', 'Swing']].head()

# Define Whiff (swinging strike)
df_joined['Whiff'] = (df_joined['event_category'] == 'swinging_strike').astype(int)

# Create squared delta_run_exp column
df_joined['delta_run_exp_squared'] = df_joined['delta_run_exp'].apply(lambda x: 
    0.5 + (x - 0.5) * 0.5 if x > 0.5 else
    -0.5 + (x + 0.5) * 0.5 if x < -0.5 else
    0.2 + (x - 0.2) * 0.75 if x > 0.2 else
    -0.2 + (x + 0.2) * 0.75 if x < -0.2 else
    x * 2.5)





numeric_cols = [
    "Vel",  "delta_run_exp",
    "Extension", "HorzApprAngle", "VertApprAngle",
    "IndVertBrk", "HorzBrk", "RelZ", "RelX"
]
for col in numeric_cols:
    df_joined[col] = pd.to_numeric(df_joined[col], errors="coerce")



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df_joined is already loaded
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_joined, x='PX', y='PZ', hue='Inzone', palette='coolwarm', alpha=0.7)

plt.xlabel('PX')
plt.ylabel('PY')
plt.title('PX vs PY Colored by Inzone')
plt.legend(title='Inzone')
plt.xlim(-2.5, 2.5)
plt.show()


In [ ]:
import pandas as pd
import numpy as np

def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Feature engineering for a baseball dataset.
    """

    # # # Step 1: Ensure relevant columns are numeric
    # # numeric_columns = ["RelX", "HorzBrk", "IndVertBrk", "Vel"]  # Add other numeric columns if needed
    # # for col in numeric_columns:
    # #     if col in df.columns:
    # #         print(f"Converting '{col}' to numeric...")
    # #         df[col] = pd.to_numeric(df[col], errors="coerce")  # Convert non-numeric to NaN

    # # Drop rows with NaN in critical columns to avoid aggregation errors
    # df = df.dropna(subset=numeric_columns)  # Drop rows with missing numeric data

    # Step 2: Determine pitcher handedness
    # df_hand = (
    #     df.groupby("pitcherId", as_index=False)["RelX"].mean()
    #     .rename(columns={"RelX": "avg_RelX"})
    # )
    # df_hand["pitcher_hand"] = ndf_hand["avg_RelX"] > 0, "R", "L")

    # # Merge handedness info back
    # df = pd.merge(df, df_hand[["pitcherId", "pitcher_hand"]], on="pitcherId", how="left")

    # Step 3: Rename columns to standard references
    df = df.rename(columns={
        "Vel": "start_speed",
        "Spin": "spin_rate",
        "Extension": "extension",
        "RelZ": "z0",           # Release height
        "RelX": "x0",           # Release side
        "HorzBrk": "ax",        # Horizontal break
        "IndVertBrk": "az",     # Vertical break
        "PitchType": "pitch_type"
    })

    # Step 4: Mirror for left-handed pitchers
    df["ax"] = np.where(df["pitcherHand"] == "L", -df["ax"], df["ax"])
    df["x0"] = np.where(df["pitcherHand"] == "L", -df["x0"], df["x0"])

    # Step 5: Most-used fastball logic
    fastball_types = ["FF", "SI","FA"]
    df_fb = df[df["pitch_type"].isin(fastball_types)].copy()

    # Group by (pitcherId, pitch_type), compute means & usage count
    df_agg = (
        df_fb.groupby(["pitcherId", "pitch_type"], as_index=False)
        .agg(
            avg_fastball_speed=("start_speed", "mean"),
            avg_fastball_az=("az", "mean"),
            avg_fastball_ax=("ax", "mean"),
            count=("start_speed", "count")
        )
    )

    # Sort by usage count, then avg_fastball_speed, descending
    df_agg = df_agg.sort_values(["count", "avg_fastball_speed"], ascending=[False, False])

    # Keep only the top row (most-used & fastest) per pitcherId
    df_agg = df_agg.drop_duplicates(subset=["pitcherId"], keep="first")

    # Step 6: Merge back & compute diffs
    df = pd.merge(
        df,
        df_agg[["pitcherId", "avg_fastball_speed", "avg_fastball_az", "avg_fastball_ax"]],
        on="pitcherId",
        how="left"
    )

    df["speed_diff"] = df["start_speed"] - df["avg_fastball_speed"]
    df["az_diff"] = df["az"] - df["avg_fastball_az"]
    df["ax_diff"] = df["ax"] - df["avg_fastball_ax"]

    return df


In [ ]:
# import pandas as pd

# Assuming the `selected_columns_df` contains the necessary columns from the combined data
# You can replace this with your DataFrame variable name.
df_joined.copy()

# Step 3: Apply the feature engineering
df_joined = feature_engineering(df_joined[df_joined["pitchingTeam"] == "USD"])


# Step 4: Check the results
df_joined.head(11)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from lightgbm import LGBMRegressor
import joblib
import matplotlib.pyplot as plt

# Step 1: Add Boolean Column for Fastballs
df_joined["is_fastball"] = df_joined["pitch_type"].isin(["FF", "FA", "SI"])

# Step 2: Define Features and Target
# # Define the list of features and include 'pitcher_hand'
# features = ["pitcherAbbrevName",
#     "start_speed",
#     "spin_rate",
#     "extension",
#     "az",
#     "ax",
#     "x0",
#     "z0",
#     "speed_diff",
#     "az_diff",
#     "ax_diff",
#     "is_fastball",
# ]

# # Include 'pitcher_hand' in the DataFrame
df_train = df_joined

# Display the first few rows
df_train.head()


In [ ]:
df_train[df_train["is_fastball"] == True].head(11)


In [ ]:
import joblib
import numpy as np
import pandas as pd

# Load the model
model_path = "C:/Users/TrevorWhite/Downloads/NCAA_STUFF_PLUS_ALL.joblib"
model = joblib.load(model_path)

# Define features
features = [
    "start_speed",
    "spin_rate",
    "extension",
    "az",
    "ax",
    "x0",
    "z0",
    "speed_diff",
    "az_diff",
    "ax_diff",
    "is_fastball"
]

# Convert features to numeric, coercing errors (non-convertible values become NaN)
df_train[features] = df_train[features].apply(pd.to_numeric, errors='coerce')

# Make predictions (Raw Value = RV)
df_train["RV"] = model.predict(df_train[features])

# Apply z-score transformation & Stuff+ scaling
target_mean_2023 = 0.011532333993710725
target_std_2023  = 0.009399038486978739

df_train["RV_zscore"] = (df_train["RV"] - target_mean_2023) / target_std_2023
df_train["Stuff+"] = 100 - (df_train["RV_zscore"] * 10)

# Display the first few rows
df_train.head()


In [ ]:
df_train.head()

In [ ]:
df_joined = df_train

df_joined = df_joined.rename(columns={
    "start_speed": "Vel",
    "spin_rate": "Spin",
    "extension": "Extension",
    "z0": "RelZ",           # Release height
    "x0": "RelX",           # Release side
    "ax": "HorzBrk",        # Horizontal break
    "az": "IndVertBrk",     # Vertical break
    "pitch_type": "PitchType"
})




In [ ]:
df_joined = df_joined.rename(columns={
    "start_speed": "Relspeed",
    "spin_rate": "Spinrate",
    "extension": "Extension",
    "z0": "Relheight",           # Release height
    "x0": "Relside",           # Release side
    "ax": "Horzbreak",        # Horizontal break
    "az": "Inducedvertbreak",     # Vertical break
    "pitchTypeFull": "Taggedpitchtype",
    "HorzApprAngle": "Horzapprangle",
    "VertApprAngle": "Vertapprangle",
    "HorzRelAngle": "Horzrelangle",
    "VertRelAngle": "Vertrelangle",
    "Stuff+": "tj_stuff_plus",
    'uniqPitchId': "pitchuid",
    "PZ": "Platelocheight",
    "PX": "Platelocside"
})


In [ ]:
df_joined[df_joined['pitchResult'].str.contains("Home Run", na=False)].head(15)

In [ ]:
# # Create Event Description column combining pitch result, baserunner state, inning, outs and count
# df_joined['Event_Desc'] = df_joined.apply(lambda row: (
#     f"{'2 Run ' if row['Runs Scored'] > 0 else ''}"
#     f"{row['pitchResult']}, "
#     f"{'Bases Empty' if not (row['ManOn1st'] == 1 or row['ManOn2nd'] == 1 or row['ManOn3rd'] == 1) else 'Runners on '}"
#     f"{'1st' if row['ManOn1st'] == 1 else ''}"
#     f"{'2nd' if row['ManOn2nd'] == 1 else ''}"
#     f"{'3rd' if row['ManOn3rd'] == 1 else ''}"
#     f", {row['inn']} "
#     f"{row['balls']}-{row['strikes']} "
#     f"{row['outs']} Out"
    
# ), axis=1)

In [ ]:
print(df_joined['Event_Desc'])

In [ ]:
df_joined['gameDate'] = pd.to_datetime(df_joined['gameDate'])

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

plt.style.use('default')  # Disables all custom styles
sns.set_theme(style=None)  # or style='white'
plt.style.use('default')
plt.rcdefaults()  # Resets everything to built-in default
plt.style.use('default')

# ----------- Force Page & Axes to Pure White, Lines/Text to Navy -----------
mpl.rcParams['figure.facecolor'] = '#FFFFFF'    # Page background
mpl.rcParams['axes.facecolor']   = '#FFFFFF'    # Axes background
mpl.rcParams['savefig.facecolor'] = '#FFFFFF'   # Saved figures also have white
for param in ('text.color','axes.edgecolor','axes.labelcolor',
              'xtick.color','ytick.color','grid.color'):
    mpl.rcParams[param] = 'navy'  # All lines/static text in navy

# -------------------------------
# Helper Functions
# -------------------------------

def get_strike_zone():
    """Returns a new Rectangle patch for the strike zone."""
    return Rectangle(
        (-0.83, 1.5), 1.66, 2.1,
        edgecolor='black', facecolor='none'
    )

def get_home_plate():
    """Returns a new Polygon patch for home plate."""
    plate_vertices = [(-0.83, 0.1), (0.83, 0.1),
                      (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]
    return Polygon(
        plate_vertices,
        closed=True, linewidth=1,
        edgecolor='black', facecolor='none'
    )

def plot_count_summary_table(ax, df):
    """
    Plots a clean table in 'ax' showing 
    RV/100 & Win % for these four count categories:
      '0-0', 'Hitters', 'Pitchers', '2K'
    """
    ax.axis('off')
    categories = {
        "0-0": "count_0_0",
        "Hitters": "count_hitters",
        "Pitchers": "count_pitchers",
        "2K": "count_2k"
    }

    table_data = []
    for cat_name, cat_col in categories.items():
        sub = df[df[cat_col] == 1]
        total_pitches = len(sub)
        if total_pitches == 0:
            rv_100 = 0.0
            win_percent = 0.0
        else:
            total_rv = sub['delta_run_exp'].sum()
            total_wins = sub['win'].sum()
            rv_100 = (total_rv / total_pitches) * 100
            win_percent = (total_wins / total_pitches) * 100

        table_data.append([cat_name, f"{rv_100:.2f}", f"{win_percent:.2f}"])

    col_labels = ["Count", "RV/100", "Win %"]
    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.3, 1.2)

    for (r, c), cell in table.get_celld().items():
        cell.get_text().set_ha("center")
        cell.get_text().set_va("center")
        cell.set_linewidth(0.5)

    ax.set_title("RV/100 and Win% by Count Category", fontsize=12, pad=10)
    pos = ax.get_position()
    ax.set_position([pos.x0 + 0.1, pos.y0, pos.width, pos.height])

def plot_logo(ax, logo_path):
    """
    Displays the logo on the left side of the Axes.
    """
    ax.axis('off')
    try:
        logo = mpimg.imread(logo_path)
        ax.imshow(logo, extent=(0, 0.5, 0, 1), aspect='auto')
    except FileNotFoundError:
        ax.text(0.25, 0.5, "Logo not found", ha='center', va='center', fontsize=12)

def plot_header(ax, text_content):
    """
    Displays the header text on the right side of the Axes.
    """
    ax.axis('off')
    ax.text(0.75, 0.5, text_content, ha='left', va='center', fontsize=12)

def plot_blank(ax):
    """Leaves the Axes blank."""
    ax.axis('off')

def plot_color_bar(ax, cmap):
    """
    Plots the vertical gradient (from -0.5 to 1.5) in the specified Axes.
    The labels have been swapped: -0.5\n3-0 out, 1.5\n0-2 HR
    """
    gradient = np.linspace(-0.5, 1.5, 256).reshape(256, 1)
    norm = plt.Normalize(-0.5, 1.5)
    im = ax.imshow(gradient, aspect='auto', cmap=cmap, norm=norm, origin='lower')
    ax.set_xticks([])
    ax.set_yticks([0, 255])
    ax.set_yticklabels(["-0.5\n3-0 out", "1.5\n0-2 HR"], fontsize=8)
    ax.tick_params(axis='y', which='both', length=0)
    ax.set_title("Run Value per Pitch", fontsize=14, rotation=90, x=-0.5, y=0.5, va='center')

def plot_pitch_scatter(ax, data, cmap, norm, title=None):
    """
    Plots a PX/PZ scatter for the given data, with an optional title.
    Maintains the aspect ratio and hides ticks.

    Additionally, the top 5 points (by highest delta_run_exp_squared) are labeled in blue.
    """
    if title:
        ax.set_title(title, fontsize=16, pad=10)

    # Standard scatter for all data
    ax.scatter(
        data['PX'], data['PZ'],
        c=data['delta_run_exp_squared'], cmap=cmap, norm=norm,
        s=60, edgecolor='black'
    )

    ax.add_patch(get_strike_zone())
    ax.add_patch(get_home_plate())
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect(5/4)

    # Identify the top 5 points by delta_run_exp_squared
    top_5 = data.nlargest(5, 'delta_run_exp').copy()
    # Sort them so the highest gets label #1, next highest #2, etc.
    top_5.sort_values('delta_run_exp', ascending=False, inplace=True)
    top_5.reset_index(drop=True, inplace=True)

    # Label them 1..5
    for rank, row in top_5.iterrows():
        x_val = row['PX']
        y_val = row['PZ']
        label_str = str(rank + 1)  # e.g. '1'..'5'

        # Slight offset for better readability
        ax.text(
            x_val + 0.05,
            y_val + 0.05,
            label_str,
            color='blue',
            fontsize=10,
            fontweight='bold'
        )


# ----------------------------------------------------------------------
def compute_bottom_row_summary(df):
    """
    Computes average values for selected pitch metrics, grouped by 'type'.
    """
    format_map = {
        "P": "{:.0f}",
        "Usage%": "{:.0f}",
        "Vel": "{:.1f}",
        "MaxVel": "{:.1f}",
        "RV/100": "{:.1f}",
        "Str%": "{:.0f}",
        "Comp%": "{:.0f}",
        "zWhiff%": "{:.0f}",
        "Chase%": "{:.0f}",
        "Ext": "{:.1f}",
        "HAA": "{:.1f}",
        "VAA": "{:.1f}",
        "IVB": "{:.1f}",
        "HB": "{:.1f}",
        "RelZ": "{:.1f}",
        "RelX": "{:.1f}"
    }

    metrics = [
        "P", "Usage%", "Vel", "MaxVel", "RV/100", "Str%", "Comp%", 
        "zWhiff%", "Chase%", "Ext", "HAA", "VAA", "IVB", "HB", "RelZ", "RelX"
    ]

    grouped_data = {}
    for pitch_type, group in df.groupby('pitchTypeFull'):
        total_pitches = len(group)

        if total_pitches == 0:
            grouped_data[pitch_type] = {m: 0.0 for m in metrics}
            continue

        rv_100 = (group["delta_run_exp"].sum() / total_pitches) * 100
        strike_pct = group["Strike"].mean() * 100
        comploc_pct = group["Comploc"].mean() * 100
        in_zone_swings = group[(group["Inzone"])]
        whiffs_in_zone = in_zone_swings["Whiff"].sum()
        z_whiff_pct = (whiffs_in_zone / len(in_zone_swings) * 100) if len(in_zone_swings) > 0 else 0.0

        out_of_zone_pitches = group[~group["Inzone"]]
        out_of_zone_swings = out_of_zone_pitches["Swing"].sum()
        chase_pct = (out_of_zone_swings / len(out_of_zone_pitches) * 100) if len(out_of_zone_pitches) > 0 else 0.0

        grouped_data[pitch_type] = {
            "P": total_pitches,
            "Usage%": (total_pitches / len(df)) * 100,
            "Vel": group["Vel"].mean(),
            "MaxVel": group["Vel"].max(),
            "RV/100": rv_100,
            "Str%": strike_pct,
            "Comp%": comploc_pct,
            "zWhiff%": z_whiff_pct,
            "Chase%": chase_pct,
            "Ext": group["Extension"].mean(),
            "HAA": group["HorzApprAngle"].mean(),
            "VAA": group["VertApprAngle"].mean(),
            "IVB": group["IndVertBrk"].mean(),
            "HB": group["HorzBrk"].mean(),
            "RelZ": group["RelZ"].mean(),
            "RelX": group["RelX"].mean()
        }

    row_labels = sorted(grouped_data.keys(), key=lambda x: grouped_data[x]["P"], reverse=True)
    cellText = []
    for pitch_type in row_labels:
        row_dict = grouped_data[pitch_type]
        row_formatted = [
            format_map[m].format(row_dict[m]) if row_dict[m] is not None else "-"
            for m in metrics
        ]
        cellText.append(row_formatted)

    return row_labels, metrics, cellText

def plot_bottom_row_table(ax, df):
    """Plots the bottom-row table grouped by 'type' (or 'pitchgroup')."""
    ax.axis('off')
    row_labels, col_labels, cellText = compute_bottom_row_summary(df)
    table = ax.table(
        cellText=cellText,
        rowLabels=row_labels,
        colLabels=col_labels,
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)

    for col_index in range(len(col_labels)):
        table.auto_set_column_width(col_index)

    table.scale(1.8, 1.8)
    for (r, c), cell in table.get_celld().items():
        cell.get_text().set_ha("center")
        cell.get_text().set_va("center")
        cell.set_linewidth(0.5)

    ax.set_title("Pitch Metrics", fontsize=14, pad=2, y=0.85)

def plot_top_5_events(ax, df):
    """
    Plots the top 5 most significant events by absolute delta_run_exp 
    in descending order. 
    - Negative values (good for pitcher) are green.
    - Positive values (bad for pitcher) are red.
    """
    ax.axis('off')
    top_events = df[['Event_Desc', 'delta_run_exp']].copy()
    top_events['abs_delta_run_exp'] = top_events['delta_run_exp'].abs()
    top_events = top_events.sort_values(by='abs_delta_run_exp', ascending=False).head(5)

    lines = []
    for i, (_, row) in enumerate(top_events.iterrows(), 1):
        color = 'green' if row['delta_run_exp'] < 0 else 'darkred'
        event_text = f"#{i}. {row['Event_Desc']}: {row['delta_run_exp']:.2f} RV"
        lines.append((event_text, color))

    y_pos = 0.8
    for text, color in lines:
        ax.text(0.025, y_pos, text, ha='left', va='center', fontsize=9, color=color)
        y_pos -= 0.13

    ax.set_title("5 Biggest Pitches", fontsize=12, pad=5, x=0.85)

def compute_game_summary(df):
    """
    Computes a summary of key pitching stats:
      - Total Pitches
      - Strike-Ball Count (concat "Strikes-Balls")
      - Strikeouts
      - Walks
      - Hits
      - Total Runs Allowed
      - First Pitch Strike % (pitchNumInAB == 1 & Strike == 1)
      - Total Whiffs
    """
    total_pitches = len(df)
    total_strikes = df['Strike'].sum()
    total_balls = df['event_category'].isin(["ball", "walk", "hit_by_pitch"]).sum()
    strikeouts = df['event_category'].eq("strikeout").sum()
    walks = df['event_category'].eq("walk").sum()
    hits = df['event_category'].isin(["single", "double", "triple", "home_run"]).sum()
    runs_allowed = int(df['Runs Scored'].sum())

    first_pitch_total = df[df["pitchNumInAB"] == 1]
    first_pitch_strike_pct = (first_pitch_total["Strike"].mean() * 100) if not first_pitch_total.empty else 0.0

    total_whiffs = df["Whiff"].sum()

    summary = {
        "P": total_pitches,
        "K-B": f"{total_strikes}-{total_balls}",
        "K": strikeouts,
        "BB": walks,
        "Hits": hits,
        "Runs": runs_allowed,
        "FPStr%": f"{first_pitch_strike_pct:.0f}%",
        "Whiffs": total_whiffs
    }
    return summary

def plot_game_summary(ax, df):
    """
    Plots the computed game summary in a structured table format inside the given Axes.
    """
    ax.axis('off')
    summary = compute_game_summary(df)
    col_labels = list(summary.keys())
    cellText = [[str(value) for value in summary.values()]]

    table = ax.table(
        cellText=cellText,
        colLabels=col_labels,
        loc="right"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)

    table.scale(2.6, 1.5)
    for (r, c), cell in table.get_celld().items():
        cell.get_text().set_ha("center")
        cell.get_text().set_va("center")
        cell.set_linewidth(0.5)

# -------------------------------
# Main Plot Layout
# -------------------------------
df_transformed = df_joined[df_joined['pitcherAbbrevName'] == 'A. Beltre']

fig = plt.figure(figsize=(12, 14))
fig.patch.set_facecolor('white')  # Force figure patch to white

height_ratios = [0.13, 0.15, 0.3, 0.3, 0.25]
width_ratios = [0.7, .8, 2.55, 2.55, 2.55]
gs = GridSpec(
    nrows=5, ncols=5, figure=fig,
    height_ratios=height_ratios,
    width_ratios=width_ratios
)

# Row 1: Logo (col 0)
ax_logo = fig.add_subplot(gs[0, 0:2])
ax_logo.set_facecolor('white')  # Force Axes face to white
plot_logo(ax_logo, r"C:\Users\TrevorWhite\Downloads\San_Diego_Toreros_logo.svg.png")

# Row 2: Header text (cols 0:2)
ax_header = fig.add_subplot(gs[1, 0:2])
ax_header.set_facecolor('white')
header_text = f"{df_transformed['pitcherAbbrevName'].iloc[0]}\nPost-Series Pitching Report\n{', '.join(df_transformed['gameDate'].dt.strftime('%Y-%m-%d').unique())}"
ax_header.text(0, 0.5, header_text, ha='left', va='center', fontsize=18)
ax_header.axis('off')

ax3 = fig.add_subplot(gs[0, 4])
ax3.set_facecolor('white')
plot_count_summary_table(ax3, df_transformed)

ax35 = fig.add_subplot(gs[0,2])
ax35.set_facecolor('white')
plot_top_5_events(ax35, df_transformed)

# Row 2: Game Summary (cols 2:3) replaces blank
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('white')
plot_game_summary(ax4, df_transformed)

# Rows 3 and 4: Color Bar (col 0) + Scatter Plots
ax5 = fig.add_subplot(gs[2:4, 0])
ax5.set_facecolor('white')
plot_color_bar(ax5, plt.cm.RdYlGn_r)

ax6 = fig.add_subplot(gs[2, 1])
ax6.set_facecolor('white')
ax6.text(0.5, 0.5, "vs RHH", ha='center', va='center', fontsize=14)
ax6.axis('off')

ax10 = fig.add_subplot(gs[3, 1])
ax10.set_facecolor('white')
ax10.text(0.5, 0.5, "vs LHH", ha='center', va='center', fontsize=14)
ax10.axis('off')

pitch_groups = {"Fast": "Fast", "Break": "Break", "Slow": "Slow"}
custom_cmap = plt.cm.RdYlGn_r
norm = TwoSlopeNorm(vmin=-0.5, vcenter=0.05, vmax=1.5)

# Row 3, columns 3-5 (RHH); Row 4, columns 3-5 (LHH)
for i, (pitch_label, pitch_group) in enumerate(pitch_groups.items()):
    ax_pitch_rhh = fig.add_subplot(gs[2, i+2])
    ax_pitch_rhh.set_facecolor('white')
    data_rhh = df_transformed[
        (df_transformed['pitchgroup'] == pitch_group) &
        (df_transformed['batterHand'] == 'R')
    ]
    plot_pitch_scatter(ax_pitch_rhh, data_rhh, custom_cmap, norm, title=pitch_label)

    ax_pitch_lhh = fig.add_subplot(gs[3, i+2])
    ax_pitch_lhh.set_facecolor('white')
    data_lhh = df_transformed[
        (df_transformed['pitchgroup'] == pitch_group) &
        (df_transformed['batterHand'] == 'L')
    ]
    plot_pitch_scatter(ax_pitch_lhh, data_lhh, custom_cmap, norm)

# Row 5: Blank
ax14 = fig.add_subplot(gs[4, :])
ax14.set_facecolor('white')
plot_blank(ax14)

# Table grouped by 'type' in bottom row
plot_bottom_row_table(ax14, df_transformed)

# After subplots are created, forcibly set each Axes face to white
for ax in fig.get_axes():
    ax.set_facecolor('white')

plt.subplots_adjust(hspace=0.05)
# Save the figure as a PDF in your Downloads folder
plt.savefig(r"C:\Users\TrevorWhite\Downloads\Pitching_Report.pdf", format="pdf", bbox_inches="tight")

# Show the figure
plt.show()




In [ ]:
import matplotlib.pyplot as plt

# Filter data where pitchResult contains "swinging", Inzone is True, Whiff is greater than 0, and pitcherAbbrevName is "A. Smith"
df_swinging = df_joined[
    (df_joined["pitchResult"].str.contains("Swinging", na=False)) & 
#     (df_joined["event_category"] != "swinging_strike") &  # Fixed: Ensure this is properly formatted
    # (df_joined["Inzone"]) &  
    # (df_joined["Whiff"] > 0) &  
    (df_joined["pitcherAbbrevName"] == "A. Smith")  # Changed to A. Smith
]

# df_swinging.head()

# Create scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(df_swinging["PX"], df_swinging["PZ"], color='blue', alpha=0.7)

# Labels and title
plt.xlabel("PX")
plt.ylabel("PZ")
plt.title("Swinging Strike Locations for A. Smith")

# Show plot
plt.show()


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle, Polygon
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
from datetime import datetime, timedelta





plt.style.use('default')  # Disables all custom styles
sns.set_theme(style=None)  # or style='white'
plt.style.use('default')
plt.rcdefaults()  # Resets everything to built-in default
plt.style.use('default')

# ----------- Force Page & Axes to Pure White, Lines/Text to Navy -----------
mpl.rcParams['figure.facecolor'] = '#FFFFFF'    # Page background
mpl.rcParams['axes.facecolor']   = '#FFFFFF'      # Axes background
mpl.rcParams['savefig.facecolor'] = '#FFFFFF'     # Saved figures also have white
for param in ('text.color','axes.edgecolor','axes.labelcolor',
              'xtick.color','ytick.color','grid.color'):
    mpl.rcParams[param] = 'navy'  # All lines/static text in navy

# -------------------------------
# Helper Functions
# -------------------------------



# Load the percentile reference CSV (ensure it's in your working directory)
percentile_table = pd.read_csv("pitch_metric_percentiles.csv")

def get_dynamic_norm(pitch_type, metric):
    """
    Given a pitch_type and a metric key (from the percentile CSV),
    return a TwoSlopeNorm using the 10th, 50th, and 90th percentile values.
    """
    # Filter rows for the given pitch type
    rows = percentile_table[percentile_table["pitch_type"] == pitch_type]
    if rows.empty:
        # Default fallback norm if no data found
        return TwoSlopeNorm(vmin=0, vcenter=50, vmax=100)
    try:
        vmin = float(rows.loc[rows["percentile"] == 0.10, metric].values[0])
        vcenter = float(rows.loc[rows["percentile"] == 0.50, metric].values[0])
        vmax = float(rows.loc[rows["percentile"] == 0.90, metric].values[0])
    except IndexError:
        return TwoSlopeNorm(vmin=0, vcenter=50, vmax=100)
    return TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

def get_strike_zone():
    """Returns a new Rectangle patch for the strike zone."""
    return Rectangle(
        (-0.83, 1.5), 1.66, 2.1,
        edgecolor='black', facecolor='none'
    )

def get_home_plate():
    """Returns a new Polygon patch for home plate."""
    plate_vertices = [(-0.83, 0.1), (0.83, 0.1),
                      (0.65, 0.25), (0, 0.5), (-0.65, 0.25)]
    return Polygon(
        plate_vertices,
        closed=True, linewidth=1,
        edgecolor='black', facecolor='none'
    )

# Load the CSV we just exported
df_count_summary = pd.read_csv("count_summary_table.csv")

def plot_count_summary_table(ax, df):
    """
    Plots a clean table in 'ax' showing 
    RV/100, Win %, and Total Pitches (P) for these four count categories:
      '0-0', 'Hitters', 'Pitchers', '2K'
    The vcenter values for RV/100 and Win % are dynamically loaded from df_count_summary.
    """
    ax.axis('off')

    categories = {
        "0-0": "count_0_0",
        "Hitters": "count_hitters",
        "Pitchers": "count_pitchers",
        "2K": "count_2k"
    }

    table_data = []
    for cat_name, cat_col in categories.items():
        sub = df[df[cat_col] == 1]
        total_pitches = len(sub)
        if total_pitches == 0:
            rv_100 = 0.0
            win_percent = 0.0
        else:
            total_rv = sub['delta_run_exp'].sum()
            total_wins = sub['win'].sum()
            rv_100 = (total_rv / total_pitches) * 100
            win_percent = (total_wins / total_pitches) * 100

        table_data.append([cat_name, f"{rv_100:.1f}", f"{win_percent:.0f}", total_pitches])

    col_labels = ["Count", "RV/100", "Win %", "P"]
    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.3, 1.2)

    # Identify column indices for RV/100 and Win %
    rv_col_idx = col_labels.index("RV/100")
    win_col_idx = col_labels.index("Win %")

    # Apply dynamic TwoSlopeNorm for each count type
    for r, row in enumerate(table_data):
        count_type = row[0]  # "0-0", "Hitters", "Pitchers", "2K"

        # Get vcenter values from df_count_summary
        row_data = df_count_summary[df_count_summary["Count"] == count_type]
        if row_data.empty:
            continue  # Skip if count type isn't found in CSV

        rv_vcenter = float(row_data["RV/100"].values[0])
        win_vcenter = float(row_data["Win %"].values[0])

        rv_norm = TwoSlopeNorm(vmin=-5, vcenter=rv_vcenter, vmax=10)
        win_norm = TwoSlopeNorm(vmin=0, vcenter=win_vcenter, vmax=100)

        for c, cell in table.get_celld().items(): 
            if c[0] == r + 1:  # Skip header row
                try:
                    value = float(cell.get_text().get_text())

                    # Color RV/100 column
                    if c[1] == rv_col_idx:
                        color = plt.cm.RdYlGn_r(rv_norm(value))
                        color = lighten_color(color, amount=0.7)  # Apply lightening
                        cell.set_facecolor(color)

                    # Color Win % column
                    elif c[1] == win_col_idx:
                        color = plt.cm.RdYlGn(win_norm(value))
                        color = lighten_color(color, amount=0.7)  # Apply lightening
                        cell.set_facecolor(color)

                except ValueError:
                    pass  # Ignore non-numeric cells

        ax.set_title("RV/100, Win% by Count", fontsize=12, pad=10)





def plot_logo(ax, logo_path):
    """
    Displays the logo on the left side of the Axes.
    """
    ax.axis('off')
    try:
        logo = mpimg.imread(logo_path)
        ax.imshow(logo, extent=(0, 0.5, 0, 1), aspect='auto')
    except FileNotFoundError:
        ax.text(0.25, 0.5, "Logo not found", ha='center', va='center', fontsize=12)

def plot_header(ax, text_content):
    """
    Displays the header text on the right side of the Axes.
    """
    ax.axis('off')
    ax.text(0.75, 0.5, text_content, ha='left', va='center', fontsize=12)

def plot_blank(ax):
    """Leaves the Axes blank."""
    ax.axis('off')

def plot_color_bar(ax, cmap):
    """
    Plots the vertical gradient (from -0.5 to 1.5) in the specified Axes.
    The labels have been swapped: -0.5\n3-0 out, 1.5\n0-2 HR
    """
    gradient = np.linspace(-0.5, 1.5, 256).reshape(256, 1)
    norm = plt.Normalize(-0.5, 1.5)
    im = ax.imshow(gradient, aspect='auto', cmap=cmap, norm=norm, origin='lower')
    ax.set_xticks([])
    ax.set_yticks([0, 255])
    ax.set_yticklabels(["-0.5\n3-0 out", "1.5\n0-2 HR"], fontsize=8)
    ax.tick_params(axis='y', which='both', length=0)
    ax.set_title("Run Value per Pitch", fontsize=14, rotation=90, x=-0.5, y=0.5, va='center')

def plot_pitch_scatter(ax, data, cmap, norm, title=None, overall_top_5=None):
    """
    Plots a PX/PZ scatter for the given data, with an optional title.
    Maintains the aspect ratio and hides ticks.

    Additionally, if overall_top_5 is provided (the overall top 5 pitches for the pitcher),
    it labels those points that are in the current subset using their overall ranking.
    """
    if title:
        ax.set_title(title, fontsize=16, pad=10)

    # Standard scatter for all data
    ax.scatter(
        data['PX'], data['PZ'],
        c=data['delta_run_exp_squared'], cmap=cmap, norm=norm,
        s=60, edgecolor='black'
    )

    ax.add_patch(get_strike_zone())
    ax.add_patch(get_home_plate())
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect(5/4)

    # If overall_top_5 is provided, label the points in the current subset with the overall rank.
    if overall_top_5 is not None:
        # Filter overall_top_5 to only those rows that appear in the current data subset.
        top_subset = overall_top_5[overall_top_5.index.isin(data.index)]
        for _, row in top_subset.iterrows():
            x_val = row['PX']
            y_val = row['PZ']
            overall_rank = row['overall_rank']  # Use the overall rank
            ax.text(
                x_val + 0.05,
                y_val + 0.05,
                str(overall_rank),
                color='blue',
                fontsize=10,
                fontweight='bold'
            )

def compute_bottom_row_summary(df): 
    """
    Computes average values for selected pitch metrics, grouped by 'pitchTypeFull', 
    while storing the abbreviated pitchType for color mapping.
    """
    format_map = {
        "P": "{:.0f}",
        "Usage%": "{:.0f}",
        "Vel": "{:.1f}",
        "MaxVel": "{:.1f}",
        "Stuff+": "{:.1f}",
        "RV/100": "{:.1f}",
        "Str%": "{:.0f}",
        "Comp%": "{:.0f}",
        "zWhiff%": "{:.0f}",
        "Chase%": "{:.0f}",
        "Ext": "{:.1f}",
        "HAA": "{:.1f}",
        "VAA": "{:.1f}",
        "IVB": "{:.1f}",
        "HB": "{:.1f}",
        "RelZ": "{:.1f}",
        "RelX": "{:.1f}"
    }

    metrics = [
        "P", "Usage%", "Vel", "MaxVel", "Stuff+", "RV/100", "Str%", "Comp%", 
        "zWhiff%", "Chase%", "Ext", "HAA", "VAA", "IVB", "HB", "RelZ", "RelX"
    ]

    # Dictionary to hold summary data and a mapping for abbreviated pitch types.
    grouped_data = {}
    abbrev_map = {}

    # Group by the full pitch type name so the table displays full names.
    for pitch_full, group in df.groupby('pitchTypeFull'):
        total_pitches = len(group)
        # Store the abbreviated pitch type from the first row.
        pitch_abbrev = group["pitchType"].iloc[0] if not group.empty else pitch_full
        abbrev_map[pitch_full] = pitch_abbrev

        if total_pitches == 0:
            grouped_data[pitch_full] = {m: 0.0 for m in metrics}
            continue

        avg_stuff_plus = group["Stuff+"].mean()
        rv_100 = (group["delta_run_exp"].sum() / total_pitches) * 100
        strike_pct = group["Strike"].mean() * 100
        comploc_pct = group["Comploc"].mean() * 100

        in_zone_swings = group[group["Inzone"]]
        whiffs_in_zone = in_zone_swings["Whiff"].sum()
        z_whiff_pct = (whiffs_in_zone / len(in_zone_swings) * 100) if len(in_zone_swings) > 0 else 0.0

        out_of_zone_pitches = group[~group["Inzone"]]
        out_of_zone_swings = out_of_zone_pitches["Swing"].sum()
        chase_pct = (out_of_zone_swings / len(out_of_zone_pitches) * 100) if len(out_of_zone_pitches) > 0 else 0.0

        grouped_data[pitch_full] = {
            "P": total_pitches,
            "Usage%": (total_pitches / len(df)) * 100,
            "Vel": group["Vel"].mean(),
            "MaxVel": group["Vel"].max(),
            "Stuff+": avg_stuff_plus,
            "RV/100": rv_100,
            "Str%": strike_pct,
            "Comp%": comploc_pct,
            "zWhiff%": z_whiff_pct,
            "Chase%": chase_pct,
            "Ext": group["Extension"].mean(),
            "HAA": group["HorzApprAngle"].mean(),
            "VAA": group["VertApprAngle"].mean(),
            "IVB": group["IndVertBrk"].mean(),
            "HB": group["HorzBrk"].mean(),
            "RelZ": group["RelZ"].mean(),
            "RelX": group["RelX"].mean()
        }

    # Sort the keys by total pitches for display order.
    row_labels = sorted(grouped_data.keys(), key=lambda x: grouped_data[x]["P"], reverse=True)
    cellText = []
    for pitch_full in row_labels:
        row_dict = grouped_data[pitch_full]
        row_formatted = [
            format_map[m].format(row_dict[m]) if row_dict[m] is not None else "-"
            for m in metrics
        ]
        cellText.append(row_formatted)

    return row_labels, metrics, cellText, abbrev_map

def plot_bottom_row_table(ax, df):
    """Plots the bottom-row table grouped by 'pitchTypeFull', 
    with dynamic coloring for RV/100, Stuff+ and additional metrics based on the percentile CSV."""
    ax.axis('off')
    # Retrieve the abbreviation map along with the summary.
    row_labels, col_labels, cellText, abbrev_map = compute_bottom_row_summary(df)
    table = ax.table(
        cellText=cellText,
        rowLabels=row_labels,
        colLabels=col_labels,
        loc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.8, 1.8)

    # Identify the fixed columns to color
    rv100_col_idx = col_labels.index("RV/100")
    stuff_col_idx = col_labels.index("Stuff+")

    # Define dynamic metrics mapping: table column name -> percentile CSV metric key
    dynamic_metrics = {
        "Str%": "strike_pct",
        "Comp%": "comploc_pct",
        "zWhiff%": "z_whiff_pct",
        "Chase%": "chase_pct"
    }
    # Get column indices for dynamic metrics
    dynamic_cols = {name: col_labels.index(name) for name in dynamic_metrics}

    # TwoSlopeNorm objects for fixed columns (RV/100 and Stuff+)
    rv_norm = TwoSlopeNorm(vmin=-5, vcenter=0, vmax=10)
    stuff_norm = TwoSlopeNorm(vmin=80, vcenter=100, vmax=120)

    # Apply background colors for all colored columns
    for (r, c), cell in table.get_celld().items():
        cell.get_text().set_ha("center")
        cell.get_text().set_va("center")
        cell.set_linewidth(0.5)

        # Skip header row (r == 0)
        if r > 0:
            # row_labels now holds full names, and we use abbrev_map to get the abbreviated pitch type.
            pitch_full = row_labels[r - 1]
            pitch_type = abbrev_map.get(pitch_full, pitch_full)

            try:
                value = float(cellText[r - 1][c])  # Adjust for header offset

                # For fixed columns
                if c == rv100_col_idx:
                    norm = rv_norm
                    color = plt.cm.RdYlGn_r(norm(value))
                elif c == stuff_col_idx:
                    norm = stuff_norm
                    color = plt.cm.RdYlGn(norm(value))
                # For dynamic metrics (from CSV percentiles)
                elif c in dynamic_cols.values():
                    col_name = [name for name, idx in dynamic_cols.items() if idx == c][0]
                    norm = get_dynamic_norm(pitch_type, dynamic_metrics[col_name])
                    color = plt.cm.PiYG(norm(value))
                else:
                    continue  # No color scaling for other columns

                # Lighten the color so that extremes are lightened
                color = lighten_color(color, amount=0.7)
                cell.set_facecolor(color)
            except ValueError:
                pass  # Non-numeric cell or header

    # Auto-adjust column widths after coloring
    for col_index in range(len(col_labels)):
        table.auto_set_column_width(col_index)

    ax.set_title("Pitch Metrics", fontsize=14, pad=2, y=0.90)





def plot_top_5_events(ax, df):
    """
    Plots the top 5 most significant events by absolute delta_run_exp 
    in descending order. 
    - Negative values (good for pitcher) are green.
    - Positive values (bad for pitcher) are red.
    """
    ax.axis('off')
    top_events = df[['Event_Desc', 'delta_run_exp']].copy()
    top_events['abs_delta_run_exp'] = top_events['delta_run_exp'].abs()
    top_events = top_events.sort_values(by='abs_delta_run_exp', ascending=False).head(5)

    lines = []
    for i, (_, row) in enumerate(top_events.iterrows(), 1):
        color = 'green' if row['delta_run_exp'] < 0 else 'darkred'
        event_text = f"#{i}. {row['Event_Desc']}: {row['delta_run_exp']:.2f} RV"
        lines.append((event_text, color))

    y_pos = 0.8
    for text, color in lines:
        ax.text(0.025, y_pos, text, ha='left', va='center', fontsize=9, color=color)
        y_pos -= 0.13

    ax.set_title("5 Biggest Pitches - See Location Plots", fontsize=12, pad=5, x=0.85)

def compute_game_summary(df):
    """
    Computes a summary of key pitching stats:
      - Total Pitches
      - Strike-Ball Count (concat "Strikes-Balls")
      - Strikeouts
      - Walks
      - Hits
      - Total Runs Allowed
      - First Pitch Strike % (pitchNumInAB == 1 & Strike == 1)
      - Total Whiffs
    """
    total_pitches = len(df)
    total_strikes = df['Strike'].sum()
    total_balls = df['event_category'].isin(["ball", "walk", "hit_by_pitch"]).sum()
    strikeouts = df['pitchResult'].str.contains("Strikeout", na=False).sum()
    walks = df['pitchResult'].isin(["Walk", "Hit By Pitch"]).sum()
    hits = df['event_category'].isin(["single", "double", "triple", "home_run"]).sum()
    runs_allowed = int(df['Runs Scored'].sum())

    first_pitch_total = df[df["pitchNumInAB"] == 1]
    first_pitch_strike_pct = (first_pitch_total["Strike"].mean() * 100) if not first_pitch_total.empty else 0.0

    total_whiffs = df["Whiff"].sum()

    summary = {
        "P": total_pitches,
        "K-B": f"{total_strikes}-{total_balls}",
        "K": strikeouts,
        "BB": walks,
        "Hits": hits,
        "Runs": runs_allowed,
        "FPStr%": f"{first_pitch_strike_pct:.1f}%",
        "Whiffs": total_whiffs
    }
    return summary

def plot_game_summary(ax, df):
    """
    Plots the computed game summary in a structured table format inside the given Axes.
    """
    ax.axis('off')
    summary = compute_game_summary(df)
    col_labels = list(summary.keys())
    cellText = [[str(value) for value in summary.values()]]

    table = ax.table(
        cellText=cellText,
        colLabels=col_labels,
        loc="right"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)

    table.scale(2.6, 1.5)
    for (r, c), cell in table.get_celld().items():
        cell.get_text().set_ha("center")
        cell.get_text().set_va("center")
        cell.set_linewidth(0.5)

# -------------------------------
# Loop through each unique pitcher and generate a report
# -------------------------------

# Assuming df_joined is your complete DataFrame that includes all pitchers.
for pitcher in df_joined['pitcherAbbrevName'].unique():
    # Filter the data for the current pitcher

    
    # Define the date range (last 5 days)
    five_days_ago = datetime.now() - timedelta(days=5)

    # Apply filters
    df_transformed = df_joined[
        (df_joined['pitcherAbbrevName'] == pitcher) &
        (df_joined['battingTeam'] != 'USD') &
        (pd.to_datetime(df_joined['gameDate']) >= five_days_ago)
     ]
    

    if df_transformed.empty:
        print(f"Skipping {pitcher} as no valid data is available.")
        continue  # Skip to the next pitcher
    
    # Compute the overall top 5 pitches for the current pitcher and assign overall rankings.
    # Compute the overall top 5 pitches based on abs(delta_run_exp)
    overall_top_5 = df_transformed.loc[df_transformed['delta_run_exp'].abs().nlargest(5).index].copy()

    # Sort them by actual delta_run_exp (not abs), so highest impact (positive or negative) is first
    overall_top_5.sort_values('delta_run_exp', ascending=False, inplace=True)

    # Assign overall ranking (1 to 5)
    overall_top_5['overall_rank'] = range(1, len(overall_top_5) + 1)


    # Set up the figure and gridspec layout
    fig = plt.figure(figsize=(12, 14))
    fig.patch.set_facecolor('white')  # Force figure patch to white

    height_ratios = [0.13, 0.15, 0.3, 0.3, 0.25]
    width_ratios = [0.7, .8, 2.55, 2.55, 2.55]
    gs = GridSpec(
        nrows=5, ncols=5, figure=fig,
        height_ratios=height_ratios,
        width_ratios=width_ratios
    )

    # Row 1: Logo (col 0)
    ax_logo = fig.add_subplot(gs[0, 0:2])
    ax_logo.set_facecolor('white')  # Force Axes face to white
    plot_logo(ax_logo, r"C:\Users\TrevorWhite\Downloads\San_Diego_Toreros_logo.svg.png")

    # Row 2: Header text (cols 0:2)
    ax_header = fig.add_subplot(gs[1, 0:2])
    ax_header.set_facecolor('white')
    header_text = f"{df_transformed['pitcherAbbrevName'].iloc[0]}\nPost-Series Pitching Report\n{', '.join(df_transformed['gameDate'].dt.strftime('%Y-%m-%d').unique())}"
    ax_header.text(0, 0.5, header_text, ha='left', va='center', fontsize=18)
    ax_header.axis('off')

    ax3 = fig.add_subplot(gs[0, 4])
    ax3.set_facecolor('white')
    plot_count_summary_table(ax3, df_transformed)

    ax35 = fig.add_subplot(gs[0, 2])
    ax35.set_facecolor('white')
    plot_top_5_events(ax35, df_transformed)

    # Row 2: Game Summary (cols 2:3) replaces blank
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.set_facecolor('white')
    plot_game_summary(ax4, df_transformed)

    # Rows 3 and 4: Color Bar (col 0) + Scatter Plots
    ax5 = fig.add_subplot(gs[2:4, 0])
    ax5.set_facecolor('white')
    plot_color_bar(ax5, plt.cm.RdYlGn_r)

    ax6 = fig.add_subplot(gs[2, 1])
    ax6.set_facecolor('white')
    ax6.text(0.5, 0.5, "vs RHH", ha='center', va='center', fontsize=14)
    ax6.axis('off')

    ax10 = fig.add_subplot(gs[3, 1])
    ax10.set_facecolor('white')
    ax10.text(0.5, 0.5, "vs LHH", ha='center', va='center', fontsize=14)
    ax10.axis('off')

    pitch_groups = {"Fast": "Fast", "Break": "Break", "Slow": "Slow"}
    custom_cmap = plt.cm.RdYlGn_r
    norm = TwoSlopeNorm(vmin=-0.5, vcenter=0.05, vmax=1.5)

    # Row 3, columns 3-5 (RHH); Row 4, columns 3-5 (LHH)
    for i, (pitch_label, pitch_group) in enumerate(pitch_groups.items()):
        ax_pitch_rhh = fig.add_subplot(gs[2, i+2])
        ax_pitch_rhh.set_facecolor('white')
        data_rhh = df_transformed[
            (df_transformed['pitchgroup'] == pitch_group) &
            (df_transformed['batterHand'] == 'R')
        ]
        plot_pitch_scatter(ax_pitch_rhh, data_rhh, custom_cmap, norm, title=pitch_label, overall_top_5=overall_top_5)

        ax_pitch_lhh = fig.add_subplot(gs[3, i+2])
        ax_pitch_lhh.set_facecolor('white')
        data_lhh = df_transformed[
            (df_transformed['pitchgroup'] == pitch_group) &
            (df_transformed['batterHand'] == 'L')
        ]
        plot_pitch_scatter(ax_pitch_lhh, data_lhh, custom_cmap, norm, overall_top_5=overall_top_5)

    # Row 5: Blank
    ax14 = fig.add_subplot(gs[4, :])
    ax14.set_facecolor('white')
    plot_blank(ax14)

    # Table grouped by 'type' in bottom row
    plot_bottom_row_table(ax14, df_transformed)

    # After subplots are created, forcibly set each Axes face to white
    for ax in fig.get_axes():
        ax.set_facecolor('white')

    plt.subplots_adjust(hspace=0.05)

    # Build the file name:
    # Assume 'battingTeam' is a column in df_transformed and use its first value.
    batting_team = df_transformed['battingTeam'].iloc[0] if 'battingTeam' in df_transformed.columns else "UnknownTeam"
    min_date = df_transformed['gameDate'].min().strftime('%Y-%m-%d')
    # file_name = f"{pitcher}_{batting_team}_ytd.pdf"
    file_name = f"{pitcher}_{batting_team}_{min_date}.pdf"
    save_path = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\scorecards" + "\\" + file_name

    # Save the figure as a PDF to the specified folder
    plt.savefig(save_path, format="pdf", bbox_inches="tight")
    print(file_name)
    plt.close(fig)  # Close the figure to free memory

# End of loop over pitchers


In [ ]:
for pitch_type in ["FA", "SL", "FC", "CH"]:
    norm = get_dynamic_norm(pitch_type, "chase_pct")
    print(f"{pitch_type} - vmin: {norm.vmin}, vcenter: {norm.vcenter}, vmax: {norm.vmax}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Load and process the game CSV files
# -------------------------------

# Load the CSV files
df_hitting = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\USDHITTINGFULL.csv")
df_pitching = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\USDPITCHINGFULL.csv")

# Append the datasets together
df = pd.concat([df_hitting, df_pitching], ignore_index=True)


# Show rows where 'pitchResult' contains 'sac' (case-insensitive)
sac_rows = df[df['pitchResult'].str.contains('sac', case=False, na=False)]
sac_rows


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------
# Load and process the game CSV files
# -------------------------------

# Load the CSV files
df_hitting = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\USDHITTINGFULL.csv")
df_pitching = pd.read_csv(r"C:\Users\TrevorWhite\Downloads\USDPITCHINGFULL.csv")

# Append the datasets together
df = pd.concat([df_hitting, df_pitching], ignore_index=True)

# -------------------------------
# Categorize the pitchResult into events
# -------------------------------

def categorize_event(event):
    """
    Categorize the pitchResult into a standardized event label.
    Bunt events are ignored (return None).
    Handles missing/NaN values gracefully.
    """
    if pd.isna(event):
        return None
    event_str = str(event).lower()

    if "single" in event_str:
        return "single"
    elif "double play" in event_str:
        return "field_out"
    elif "double" in event_str:
        return "double"
    elif "triple" in event_str:
        return "triple"
    elif "home run" in event_str:
        return "home_run"
    elif "looking" in event_str:
        return "called_strike"
    elif "swinging" in event_str:
        return "swinging_strike"
    elif "hit by pitch" in event_str:
        return "hit_by_pitch"
    elif "ball" in event_str:
        return "ball"
    elif "walk" in event_str:
        return "walk"
    elif "foul" in event_str:
        return "foul"
    elif "sac fly" in event_str:
        return "sac fly"
    elif "sac bunt" in event_str:
        return "sac bunt"
    elif ("line out" in event_str or "fly out" in event_str or "ground out" in event_str or 
          "pop out" in event_str or "double play" in event_str or "reached on error" in event_str or 
          "in play out" in event_str or "sac fly" in event_str or "fielder's choice" in event_str):
        return "field_out"
    elif "ball in the dirt" in event_str:
        return "ball"
    elif "bunt" in event_str:
        return "bunt"
    else:
        return "unknown"

# Apply the event categorization to the pitchResult column
df['event_category'] = df['pitchResult'].apply(categorize_event)

# Drop rows where event_category is None (e.g. bunts)
# df = df[(df['event_category'].notna()) & (df['event_category'] != 'unknown')].copy()

# -------------------------------
# Calculate Runs Scored
# -------------------------------
import numpy as np

# Initialize Runs Scored column with 0s
df['Runs Scored'] = 0

# Calculate diffs
usd_diff = (df['currentRuns'].shift(-1) - df['currentRuns'].fillna(0)).clip(lower=0)
opp_diff = (df['opponentCurrentRuns'].shift(-1) - df['opponentCurrentRuns']).clip(lower=0)

# Apply conditionally
df.loc[df['battingTeam'] == 'USD', 'Runs Scored'] = usd_diff
df.loc[df['battingTeam'] != 'USD', 'Runs Scored'] = opp_diff


# -------------------------------
# Convert gameDate to datetime
# -------------------------------
df['gameDate'] = pd.to_datetime(df['gameDate'])

# -------------------------------
# Create leadoff batter indicators
# -------------------------------
valid_leadoff_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}

# Step 1: Identify leadoff batters (first PA in each inning)
df["inning_leadoff"] = df.groupby(["gameDate", "inn"])["abNumInGame"].transform("min") == df["abNumInGame"]

# Step 2: Identify successful leadoff batters (reached base via valid event)
df["inning_leadoff"] = df["inning_leadoff"] & df["event_category"].isin(valid_leadoff_events)

# Step 3: If a leadoff batter succeeded in an inning, mark success for all rows in that inning
df["inning_leadoff_success"] = df.groupby(["gameDate", "inn"])["inning_leadoff"].transform("max")

# Convert to integer (True/False → 1/0)
df["inning_leadoff"] = df["inning_leadoff"].astype(int)
df["inning_leadoff_success"] = df["inning_leadoff_success"].astype(int)

#######give me any adv code to add here
# -------------------------------
# Any-advancement flags (with strikeout exception)
# -------------------------------

# Ensure a stable sort within game/inning/AB
order_cols = [c for c in ['gameId','inn','abNumInGame','pitchNumInGame'] if c in df.columns]
if order_cols:
    df = df.sort_values(order_cols)

# Normalize base flags to 0/1 (treat NaN as no runner)
for c in ['ManOn1st','ManOn2nd','ManOn3rd']:
    if c in df.columns:
        df[c] = df[c].fillna(0).astype(int)

# Baseline: compare to previous row WITHIN the same AB
g_ab = df.groupby([c for c in ['gameId','inn','abNumInGame'] if c in df.columns], group_keys=False)
df['p1'] = g_ab['ManOn1st'].shift(1).fillna(0).astype(int)
df['p2'] = g_ab['ManOn2nd'].shift(1).fillna(0).astype(int)
df['p3'] = g_ab['ManOn3rd'].shift(1).fillna(0).astype(int)
df['pR'] = g_ab['Runs Scored'].shift(1).fillna(0).astype(int)

adv_1_to_2_same = (df['p1'] == 1) & (df['ManOn1st'] == 0) & (df['p2'] == 0) & (df['ManOn2nd'] == 1)
adv_2_to_3_same = (df['p2'] == 1) & (df['ManOn2nd'] == 0) & (df['p3'] == 0) & (df['ManOn3rd'] == 1)
adv_3_to_home_same = (df['p3'] == 1) & (df['ManOn3rd'] == 0) & ((df['Runs Scored'] - df['pR']) >= 1)

# Strikeout exception: if the IMMEDIATELY PREVIOUS row (same game+inning) is a strikeout,
# allow cross-AB comparison using that previous row only
g_inn = df.groupby([c for c in ['gameId','inn'] if c in df.columns], group_keys=False)
prev_is_k = g_inn['pitchResult'].shift(1).astype(str).str.contains('strikeout', case=False, na=False)

df['g1'] = g_inn['ManOn1st'].shift(1).fillna(0).astype(int)
df['g2'] = g_inn['ManOn2nd'].shift(1).fillna(0).astype(int)
df['g3'] = g_inn['ManOn3rd'].shift(1).fillna(0).astype(int)
df['gR'] = g_inn['Runs Scored'].shift(1).fillna(0).astype(int)

adv_1_to_2_k = prev_is_k & (df['g1'] == 1) & (df['ManOn1st'] == 0) & (df['g2'] == 0) & (df['ManOn2nd'] == 1)
adv_2_to_3_k = prev_is_k & (df['g2'] == 1) & (df['ManOn2nd'] == 0) & (df['g3'] == 0) & (df['ManOn3rd'] == 1)
adv_3_to_home_k = prev_is_k & (df['g3'] == 1) & (df['ManOn3rd'] == 0) & ((df['Runs Scored'] - df['gR']) >= 1)

# Final flag
df['any_adv'] = adv_1_to_2_same | adv_2_to_3_same | adv_3_to_home_same | adv_1_to_2_k | adv_2_to_3_k | adv_3_to_home_k

# -------------------------------
# Total bases (custom):
# single/double/triple/HR = 1–4
# walk, HBP = 1
# sac fly, sac bunt = 1
# +1 if any_adv is True
# -------------------------------
tb_map = {
    "single": 1,
    "double": 2,
    "triple": 3,
    "home_run": 4,
    "walk": 1,
    "hit_by_pitch": 1,
    "sac fly": 1,
    "sac bunt": 1,
}

# base TB from event
df["total_bases"] = df["event_category"].map(tb_map).fillna(0).astype(int)

# add +1 if any advancement occurred


# optional: if you want to keep it bounded at 4, uncomment:
# df["total_bases"] = df["total_bases"].clip(upper=4)

# -------------------------------
# Got-on-and-scored-inning (minimal deps)
# Needs: gameId, inn, battingTeam, abNumInGame, event_category, Runs Scored
# Rule: if R runs scored in the half-inning, credit the first R qualifying PAs
# (single/double/triple/home_run/walk/hit_by_pitch). Broadcast to all pitches in that AB.
# -------------------------------
# -------------------------------
# Got-on-and-scored-inning (direct runs vs. men-on check)
# -------------------------------

# ---------------------------------------------
# runs_in_half (joins to all rows)
# man_on_in_half (ONLY on the qualifying event row)
# Qualifying events: single/double/triple/home_run/walk/hit_by_pitch
# ---------------------------------------------
qual = {"single","double","triple","home_run","walk","hit_by_pitch"}

# Stable order
order_cols = [c for c in ['gameId','inn','battingTeam','abNumInGame','pitchNumInGame'] if c in df.columns]
if order_cols:
    df = df.sort_values(order_cols)

half_keys = [c for c in ['gameId','inn','battingTeam'] if c in df.columns]

# runs_in_half: total runs in the half-inning (broadcast to all rows)
runs_half = (df.groupby(half_keys)['Runs Scored']
               .sum()
               .reset_index(name='runs_in_half'))
df = df.merge(runs_half, on=half_keys, how='left')
df['runs_in_half'] = df['runs_in_half'].fillna(0).astype(int)

# man_on_in_half: ordinal among qualifying ON-BASE events, only on that event row
# Filter to the event rows that *create* a baserunner (one per PA)
qual_rows = df[df['event_category'].isin(qual)].copy()

# Ensure deterministic ordering among qualifying events within the half-inning
if order_cols:
    qual_rows = qual_rows.sort_values(order_cols)

# Ordinal 1..N within each half-inning
qual_rows['man_on_in_half'] = (
    qual_rows.groupby(half_keys).cumcount() + 1
).astype('Int64')

# Write back ONLY to those exact event rows
df['man_on_in_half'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
df.loc[qual_rows.index, 'man_on_in_half'] = qual_rows['man_on_in_half'].astype('Int64')
# got_on_and_scored_inning:
# 1 only on qualifying on-base event rows where man_on_in_half <= runs_in_half
# 0 elsewhere (incl. non-qual rows)
df['got_on_and_scored_inning'] = 0
mask = df['man_on_in_half'].notna() & (df['man_on_in_half'] <= df['runs_in_half'])
df.loc[mask, 'got_on_and_scored_inning'] = 1

df.loc[df['got_on_and_scored_inning'] == 1, 'total_bases'] = 4


# df["total_bases"] = df["total_bases_base"]  + df["any_adv"].astype(int)


# -------------------------------
# Set df_joined for battle report compatibility
# -------------------------------
df_joined = df

print("Data loading and preparation complete for battle reports!")
print(f"DataFrame shape: {df_joined.shape}")

In [ ]:
import pandas as pd

# Display all columns and rows in DataFrame outputs
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [ ]:
# Show all rows for the specified game, sorted by total_bases descending
df_game = df[df['gameId'] == 142703823].copy()
df_game_sorted = df_game.sort_values('total_bases', ascending=False)
df_game_sorted

In [ ]:
# Top 7 subset + order
df_top7 = (
    df[(df['gameId'] == 197307673) & (df['inn'] == "Top 7")]
      .copy()
      .sort_values('abNumInGame' if 'abNumInGame' in df.columns else 'pitchNumInGame')
)

# Normalize base flags to 0/1
for c in ['ManOn1st','ManOn2nd','ManOn3rd']:
    if c in df_top7.columns:
        df_top7[c] = df_top7[c].fillna(0).astype(int)

# Normalize runs
df_top7['Runs Scored'] = df_top7['Runs Scored'].fillna(0).astype(int)

# --- Previous states within AB (baseline rule) ---
g = df_top7.groupby('abNumInGame', group_keys=False)
df_top7['p1'] = g['ManOn1st'].shift(1).fillna(0).astype(int)
df_top7['p2'] = g['ManOn2nd'].shift(1).fillna(0).astype(int)
df_top7['p3'] = g['ManOn3rd'].shift(1).fillna(0).astype(int)
df_top7['pR'] = g['Runs Scored'].shift(1).fillna(0).astype(int)

# Advancements within same AB
adv_1_to_2_same = (df_top7['p1'] == 1) & (df_top7['ManOn1st'] == 0) & (df_top7['p2'] == 0) & (df_top7['ManOn2nd'] == 1)
adv_2_to_3_same = (df_top7['p2'] == 1) & (df_top7['ManOn2nd'] == 0) & (df_top7['p3'] == 0) & (df_top7['ManOn3rd'] == 1)
adv_3_to_home_same = (df_top7['p3'] == 1) & (df_top7['ManOn3rd'] == 0) & ((df_top7['Runs Scored'] - df_top7['pR']) >= 1)

# --- Strikeout exception: allow cross-AB comparison ONLY if previous row is a strikeout ---
prev_is_k = df_top7['pitchResult'].shift(1).astype(str).str.contains('strikeout', case=False, na=False)

# Global previous row states (cross-AB)
df_top7['g1'] = df_top7['ManOn1st'].shift(1).fillna(0).astype(int)
df_top7['g2'] = df_top7['ManOn2nd'].shift(1).fillna(0).astype(int)
df_top7['g3'] = df_top7['ManOn3rd'].shift(1).fillna(0).astype(int)
df_top7['gR'] = df_top7['Runs Scored'].shift(1).fillna(0).astype(int)

adv_1_to_2_k = prev_is_k & (df_top7['g1'] == 1) & (df_top7['ManOn1st'] == 0) & (df_top7['g2'] == 0) & (df_top7['ManOn2nd'] == 1)
adv_2_to_3_k = prev_is_k & (df_top7['g2'] == 1) & (df_top7['ManOn2nd'] == 0) & (df_top7['g3'] == 0) & (df_top7['ManOn3rd'] == 1)
adv_3_to_home_k = prev_is_k & (df_top7['g3'] == 1) & (df_top7['ManOn3rd'] == 0) & ((df_top7['Runs Scored'] - df_top7['gR']) >= 1)

# Any advancement (same-AB OR strikeout-exception cross-AB)
df_top7['any_adv'] = (
    adv_1_to_2_same | adv_2_to_3_same | adv_3_to_home_same |
    adv_1_to_2_k    | adv_2_to_3_k    | adv_3_to_home_k
)

# Return row(s) for the last at-bat in Top 7 with any advancement
if df_top7['any_adv'].any():
    last_abnum = df_top7.loc[df_top7['any_adv'], 'abNumInGame'].iloc[-1]
    result = df_top7[(df_top7['abNumInGame'] == last_abnum) & (df_top7['any_adv'])]
else:
    result = df_top7.iloc[0:0]

df_top7[['abNumInGame', 'pitchNumInGame', 'ManOn1st', 'ManOn2nd', 'ManOn3rd', 'Runs Scored', 'any_adv']]


In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import os
from textwrap import fill
from matplotlib import colors

# Define performance goals based on the 2025 OFFENSE BATTLES document
GOALS = {
    "Battle 1: Leadoff Baserunners": {
        "Leadoff Runners (Offense)": 4,   # Offense goal: get to 4+ leadoff runners
        "Leadoff Runners (Defense)": 3    # Defense goal: 3 or fewer allowed
    },
    "Battle 2: Score the Leadoff Baserunners": {
        "Leadoff Runs % (Offense)": 67,     # Offense goal: 67% or higher
        "Leadoff Stranded % (Defense)": 70  # Defense goal: 70% or higher
    },
    "Battle 3: Total Baserunners": {
        "Total Baserunners (Offense)": 16,  # Offense goal: get to 16+ baserunners
        "Total BR + Extra Bases (Offense)": 24,
        "Total Baserunners (Defense)": 13   # Defense goal: 13 or fewer allowed
    },
    "Battle 4: Clean Defense": {
        "Defensive Errors (Pitching)": 0   # Goal: 0 USD errors
    },
    "Battle 5: Free 90s": {
        "BB + HBP (Offense)": "More than K",  # Offense goal: more walks/HBP than strikeouts (not directly calculated here)
        "BB + HBP (Defense)": 3              # Defense goal: 3 or fewer allowed
    }
}

def compute_battle_performance(df):
    """
    Computes battle metrics for USD, separating batting and pitching performances.
    Includes numerator and denominator for percentage calculations and 
    compares actual values to defined goals.
    """
    # Filter for USD's batting and pitching data
    batting_df = df[df["battingTeam"] == "USD"].copy()
    pitching_df = df[df["pitchingTeam"] == "USD"].copy()

    # Battle 1: Leadoff Baserunners
    total_innings_batting = batting_df["inn"].nunique()
    leadoff_success_bat = batting_df["inning_leadoff"].sum()  # offense
    leadoff_success_pitch = pitching_df["inning_leadoff"].sum()  # defense

    # Battle 2: Score the Leadoff Baserunners (Offense)
    if leadoff_success_bat > 0:
        grouped_inn_bat = batting_df.groupby("inn").agg({
            "inning_leadoff": "max",
            "Runs Scored": "max"
        }).reset_index()
        innings_with_runs = grouped_inn_bat[
            (grouped_inn_bat["inning_leadoff"] == 1) & (grouped_inn_bat["Runs Scored"] > 0)
        ].shape[0]
        leadoff_runs_pct = (innings_with_runs / leadoff_success_bat) * 100
        leadoff_runs_text = f"{innings_with_runs}/{leadoff_success_bat} ({leadoff_runs_pct:.1f}%)"
        leadoff_runs_met = leadoff_runs_pct >= GOALS["Battle 2: Score the Leadoff Baserunners"]["Leadoff Runs % (Offense)"]
    else:
        leadoff_runs_text, leadoff_runs_met = "0/0 (0.0%)", False

    # Battle 2: Score the Leadoff Baserunners (Defense)
    if leadoff_success_pitch > 0:
        grouped_inn_pitch = pitching_df.groupby("inn").agg({
            "inning_leadoff": "max",
            "Runs Scored": "max"
        }).reset_index()
        innings_with_runs_allowed = grouped_inn_pitch[
            (grouped_inn_pitch["inning_leadoff"] == 1) & (grouped_inn_pitch["Runs Scored"] > 0)
        ].shape[0]
        innings_leadoff_stranded = leadoff_success_pitch - innings_with_runs_allowed
        leadoff_stranded_pct = (innings_leadoff_stranded / leadoff_success_pitch) * 100
        leadoff_stranded_text = f"{innings_leadoff_stranded}/{leadoff_success_pitch} ({leadoff_stranded_pct:.1f}%)"
        leadoff_stranded_met = leadoff_stranded_pct >= GOALS["Battle 2: Score the Leadoff Baserunners"]["Leadoff Stranded % (Defense)"]
    else:
        leadoff_stranded_text, leadoff_stranded_met = "0/0 (0.0%)", False

    # Battle 3: Total Baserunners
    baserunner_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}
    total_baserunners_off = batting_df[batting_df["event_category"].isin(baserunner_events)].shape[0]
    total_baserunners_def = pitching_df[pitching_df["event_category"].isin(baserunner_events)].shape[0]
    tb_off = int(batting_df['total_bases'].fillna(0).sum()) if 'total_bases' in batting_df.columns else 0
    xb_off = int(batting_df['any_adv'].fillna(0).sum()) if 'any_adv' in batting_df.columns else 0
    tb_plus_xb_off = tb_off + xb_off

    # Battle 4: Clean Defense
    defensive_errors = pitching_df["pitchResult"].str.contains("error", case=False, na=False).sum()

    # Battle 5: Free 90s
    free_90s_off = batting_df[batting_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0]
    strikeouts_off = batting_df['pitchResult'].str.contains("Strikeout", na=False).sum()
    free_90s_def = pitching_df[pitching_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0]

    return {
        "Battle 1: Leadoff Baserunners": {
            "Leadoff Runners (Offense)": f"{leadoff_success_bat} (Goal: 4+) {'✔' if leadoff_success_bat >= 4 else '✘'}",
            "Leadoff Runners (Defense)": f"{leadoff_success_pitch} (Goal: ≤3) {'✔' if leadoff_success_pitch <= 3 else '✘'}"
        },
        "Battle 2: Score the Leadoff Baserunners": {
            "Leadoff Runs % (Offense)": f"{leadoff_runs_text} (Goal: 67%+) {'✔' if leadoff_runs_met else '✘'}",
            "Leadoff Stranded % (Defense)": f"{leadoff_stranded_text} (Goal: 70%+) {'✔' if leadoff_stranded_met else '✘'}"
        },
        "Battle 3: Total Baserunners": {
            "Total Baserunners (Offense)": f"{total_baserunners_off} (Goal: 16+) {'✔' if total_baserunners_off >= 16 else '✘'}",
            "Total BR + Extra Bases (Offense)": f"{tb_plus_xb_off} (Goal: 24+) {'✔' if tb_plus_xb_off >= 24 else '✘'}",
            "Total Baserunners (Defense)": f"{total_baserunners_def} (Goal: ≤13) {'✔' if total_baserunners_def <= 13 else '✘'}"
        },
        "Battle 4: Clean Defense": {
            "Defensive Errors (Pitching)": f"{defensive_errors} (Goal: 0) {'✔' if defensive_errors == 0 else '✘'}"
        },
        "Battle 5: Free 90s": {
            "BB + HBP (Offense)": f"{free_90s_off} (Goal: More than K ({strikeouts_off})) {'✔' if free_90s_off > strikeouts_off else '✘'}",
            "BB + HBP (Defense)": f"{free_90s_def} (Goal: ≤3) {'✔' if free_90s_def <= 3 else '✘'}"
        }
    }

def create_battle_report_text(df):
    """
    Computes the battle performance report and returns it as a formatted string.
    """
    battle_results = compute_battle_performance(df)
    game_date = df['gameDate'].iloc[0].strftime('%Y-%m-%d')
    
    report_lines = [f"=== USD Battle Performance Report (Battles 1–5) - {game_date} ===\n"]
    for battle_name, metrics in battle_results.items():
        report_lines.append(battle_name)
        for metric_label, metric_value in metrics.items():
            report_lines.append(f"   {metric_label}: {metric_value}")
        report_lines.append("")
    return "\n".join(report_lines)

def compute_battle_metrics_table(df):
    """
    Returns per-metric display strings like '3/4' (or similar) and met flags for coloring,
    plus components needed to compute season summaries.
    Output dict keys are canonical metric codes matching table columns.
    """
    batting_df = df[df["battingTeam"] == "USD"].copy()
    pitching_df = df[df["pitchingTeam"] == "USD"].copy()

    out = {}

    # Battle 1: Leadoff Runners
    leadoff_off = int(batting_df["inning_leadoff"].sum())
    leadoff_def = int(pitching_df["inning_leadoff"].sum())
    goal_1a = 4
    goal_1b = 3
    out["B1a Leadoff Runners (Off)"] = {
        "display": f"{leadoff_off}/{goal_1a}",
        "met": leadoff_off >= goal_1a,
        "value": leadoff_off,
        "goal": goal_1a,
        "vs_goal_pct": (leadoff_off / goal_1a * 100.0) if goal_1a else np.nan
    }
    out["B1b Leadoff Runners (Def)"] = {
        "display": f"{leadoff_def}/{goal_1b}",
        "met": leadoff_def <= goal_1b,
        "value": leadoff_def,
        "goal": goal_1b,
        "vs_goal_pct": (goal_1b / leadoff_def * 100.0) if leadoff_def > 0 else 100.0
    }

    # Battle 2: Leadoff scored / stranded
    if leadoff_off > 0:
        grp_b = batting_df.groupby("inn").agg({"inning_leadoff": "max", "Runs Scored": "max"}).reset_index()
        innings_with_runs = grp_b[(grp_b["inning_leadoff"] == 1) & (grp_b["Runs Scored"] > 0)].shape[0]
        pct_off = (innings_with_runs / leadoff_off) * 100.0
        display_off = f"{innings_with_runs}/{leadoff_off}"
    else:
        innings_with_runs, pct_off, display_off = 0, 0.0, "0/0"

    if leadoff_def > 0:
        grp_p = pitching_df.groupby("inn").agg({"inning_leadoff": "max", "Runs Scored": "max"}).reset_index()
        runs_allowed = grp_p[(grp_p["inning_leadoff"] == 1) & (grp_p["Runs Scored"] > 0)].shape[0]
        stranded = leadoff_def - runs_allowed
        pct_def = (stranded / leadoff_def) * 100.0
        display_def = f"{stranded}/{leadoff_def}"
    else:
        stranded, pct_def, display_def = 0, 0.0, "0/0"

    out["B2a Leadoff Runs % (Off)"] = {
        "display": display_off,  # numerator/denominator
        "met": pct_off >= 67.0,
        "value": pct_off,
        "goal": 67.0,
        "vs_goal_pct": (pct_off / 67.0 * 100.0) if 67.0 else np.nan
    }
    out["B2b Leadoff Stranded % (Def)"] = {
        "display": display_def,  # stranded/total leadoff reached
        "met": pct_def >= 70.0,
        "value": pct_def,
        "goal": 70.0,
        "vs_goal_pct": (pct_def / 70.0 * 100.0) if 70.0 else np.nan
    }

    # Battle 3: Total Baserunners
    baserunner_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}
    tot_off = int(batting_df[batting_df["event_category"].isin(baserunner_events)].shape[0])
    tot_def = int(pitching_df[pitching_df["event_category"].isin(baserunner_events)].shape[0])
    goal_3a, goal_3b = 16, 13
    out["B3a Total Baserunners (Off)"] = {
        "display": f"{tot_off}/{goal_3a}",
        "met": tot_off >= goal_3a,
        "value": tot_off,
        "goal": goal_3a,
        "vs_goal_pct": (tot_off / goal_3a * 100.0)
    }
    out["B3b Total Baserunners (Def)"] = {
        "display": f"{tot_def}/{goal_3b}",
        "met": tot_def <= goal_3b,
        "value": tot_def,
        "goal": goal_3b,
        "vs_goal_pct": (goal_3b / tot_def * 100.0) if tot_def > 0 else 100.0
    }
    tb_off = int(batting_df['total_bases'].fillna(0).sum()) if 'total_bases' in batting_df.columns else 0
    xb_off = int(batting_df['any_adv'].fillna(0).sum()) if 'any_adv' in batting_df.columns else 0
    tb_plus_xb_off = tb_off + xb_off

    out["B3c Total Bases + XBs (Off)"] = {
        "display": f"{tb_plus_xb_off}/24",
        "met": tb_plus_xb_off >= 24,
        "value": tb_plus_xb_off,
        "goal": 24,
        "vs_goal_pct": (tb_plus_xb_off / 24 * 100.0)
    }


    ##also add in here something to count occurences where any adv is true

    # Battle 4: Errors
    errors = int(pitching_df["pitchResult"].str.contains("error", case=False, na=False).sum())
    out["B4 Defensive Errors (Pitch)"] = {
        "display": f"{errors}/0",
        "met": errors == 0,
        "value": errors,
        "goal": 0.0,
        "vs_goal_pct": 100.0 if errors == 0 else 0.0
    }

    # Battle 5: Free 90s
    free90_off = int(batting_df[batting_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0])
    ks_off = int(batting_df['pitchResult'].str.contains("Strikeout", na=False).sum())
    free90_def = int(pitching_df[pitching_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0])

    out["B5a BB+HBP (Off) vs K"] = {
        "display": f"{free90_off}/{ks_off}",
        "met": free90_off > ks_off,
        "value": free90_off,
        "goal": float(ks_off),
        "vs_goal_pct": (free90_off / max(ks_off, 1) * 100.0)
    }
    out["B5b BB+HBP (Def)"] = {
        "display": f"{free90_def}/3",
        "met": free90_def <= 3,
        "value": free90_def,
        "goal": 3.0,
        "vs_goal_pct": (3.0 / free90_def * 100.0) if free90_def > 0 else 100.0
    }

    return out

def render_season_summary_pdf(rows_df, summary_df, out_path):
    """
    Render a season-long combined PDF with two pages:
      - Page 1: game-by-game table
      - Page 2: totals summary table
    """
    # Column order & labels
    metric_cols = [
        "B1a Leadoff Runners (Off)",
        "B1b Leadoff Runners (Def)",
        "B2a Leadoff Runs % (Off)",
        "B2b Leadoff Stranded % (Def)",
        "B3a Total Baserunners (Off)",
        "B3c Total Bases + XBs (Off)",
        "B3b Total Baserunners (Def)",
        "B4 Defensive Errors (Pitch)",
        "B5a BB+HBP (Off) vs K",
        "B5b BB+HBP (Def)",
    ]
    display_cols = ["GameDate", "Opponent", "Score"] + metric_cols

    # Wrap headers for natural spacing
    wrapped_headers = [fill(h, width=14) for h in display_cols]

    # --- Page 1: per-game table ---
    disp = rows_df[display_cols].copy()

    fig1 = plt.figure(figsize=(17, 11))
    fig1.patch.set_facecolor("white")
    plt.suptitle("USD Battles Season Summary - Game by Game", fontsize=18, y=0.98)
    ax1 = plt.axes([0.02, 0.08, 0.96, 0.86])
    ax1.axis("off")

    # --- CONTINUOUS COLOR SCALE LOGIC ---
    def get_color_for_pct(pct, met, is_error_metric=False):
        if pct is None or np.isnan(pct):
            return "white"
        # Lighter color scales for better readability
        if met:
            # Passing: very light green to medium green
            norm = min(max(pct, 0), 100) / 100.0
            # "#eafaf1" (very light green) to "#6fcf97" (medium green)
            cmap = colors.LinearSegmentedColormap.from_list("", ["#eafaf1", "#6fcf97"])
            return colors.to_hex(cmap(norm))
        else:
            # Failing: very light red/pink to medium red
            norm = min(max(pct, 0), 100) / 100.0
            # "#fdeaea" (very light red) to "#eb5757" (medium red)
            cmap = colors.LinearSegmentedColormap.from_list("", ["#fdeaea", "#eb5757"])
            return colors.to_hex(cmap(norm))

    cell_colours = [["white"] * len(display_cols) for _ in range(len(disp))]
    for r_idx, row in rows_df.iterrows():
        for c_idx, col in enumerate(display_cols):
            if col in ["GameDate", "Opponent", "Score"]:
                cell_colours[r_idx][c_idx] = "white"
            else:
                vs_goal_col = col + "__vs_goal_pct"
                met_col = col + "__met"
                is_error_metric = col == "B4 Defensive Errors (Pitch)"
                pct = row.get(vs_goal_col, None)
                met = row.get(met_col, False)
                if pct is None and met_col in row:
                    cell_colours[r_idx][c_idx] = "#c6efce" if met else "#ffc7ce"
                else:
                    cell_colours[r_idx][c_idx] = get_color_for_pct(pct, met, is_error_metric=is_error_metric)

    table_data = [wrapped_headers] + disp.values.tolist()
    colours = [["#d9d9d9"] * len(display_cols)] + cell_colours

    table = ax1.table(cellText=table_data,
                      cellLoc='center',
                      cellColours=colours,
                      loc='upper left')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1.0, 1.4)

    # Bold header row and set header row height larger
    for c in range(len(display_cols)):
        cell = table[0, c]
        cell.set_text_props(fontweight='bold')
        cell.set_height(0.08)  # Increase header row height

    # --- Page 2: totals summary ---
    fig2 = plt.figure(figsize=(14, 8))
    fig2.patch.set_facecolor("white")
    plt.suptitle("USD Battles Season Totals Summary", fontsize=18, y=0.95)
    ax2 = plt.axes([0.02, 0.08, 0.96, 0.85])
    ax2.axis("off")

    sum_headers = ["Metric", "Games Met", "Avg % of Goal"]
    sum_rows = []
    for metric in metric_cols:
        row = summary_df.loc[summary_df["Metric"] == metric]
        if not row.empty:
            games = int(row["Games"].values[0])
            met = int(row["Games_Met"].values[0])
            avg_pct = row["Avg_Vs_Goal_Pct"].values[0]
            sum_rows.append([fill(metric, width=25), f"{met}/{games}", f"{avg_pct:.1f}%"])
        else:
            sum_rows.append([fill(metric, width=25), "0/0", "—"])

    sum_table_data = [sum_headers] + sum_rows
    sum_colours = [["#d9d9d9", "#d9d9d9", "#d9d9d9"]] + [["white"] * 3 for _ in sum_rows]

    sum_table = ax2.table(cellText=sum_table_data,
                          cellLoc='center',
                          cellColours=sum_colours,
                          loc='upper left')
    sum_table.auto_set_font_size(False)
    sum_table.set_fontsize(10)
    sum_table.scale(1.0, 1.4)

    # Increase row height for all rows (not just header)
    for r in range(len(sum_table_data)):
        for c in range(len(sum_headers)):
            cell = sum_table[r, c]
            if r == 0:
                cell.set_text_props(fontweight='bold')
                cell.set_height(0.08)
            else:
                cell.set_height(0.06)  # Increase row height for summary rows

    # Save both pages into single PDF
    from matplotlib.backends.backend_pdf import PdfPages
    with PdfPages(out_path) as pdf:
        pdf.savefig(fig1, bbox_inches="tight", facecolor='white')
        pdf.savefig(fig2, bbox_inches="tight", facecolor='white')
    plt.close(fig1)
    plt.close(fig2)

# Loop through each unique gameId and generate per-game PDFs (unchanged)
scorecards_folder = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\scorecards"
os.makedirs(scorecards_folder, exist_ok=True)

season_rows = []
season_summ_rows = []

ordered_ids = df_joined.groupby("gameId")["gameDate"].min().sort_values(ascending=True).index
for game_id in ordered_ids:
    game_df = df_joined[df_joined["gameId"] == game_id]
    game_df = game_df.sort_values("gameDate")
    if game_df.empty:
        continue
    game_date = game_df['gameDate'].iloc[0].strftime('%Y-%m-%d')
    report_text = create_battle_report_text(game_df)
    
    fig, ax = plt.subplots(figsize=(8.5, 11), facecolor='white')
    ax.set_facecolor('white')
    ax.axis("off")
    ax.text(0.05, 0.95, report_text, transform=ax.transAxes, fontsize=14, va="top", family="monospace", color="black")
    
    file_name = f"BattleReport_{game_date}_{game_id}.pdf"
    save_path = scorecards_folder + "\\" + file_name
    fig.savefig(save_path, format="pdf", bbox_inches="tight", facecolor='white')
    plt.close(fig)

    # Get totalRuns, opponentRuns, and opponent for this game
    # Assume these columns exist and are constant for the game
    total_runs = game_df["totalRuns"].iloc[0] if "totalRuns" in game_df.columns else np.nan
    opponent_runs = game_df["opponentRuns"].iloc[0] if "opponentRuns" in game_df.columns else np.nan
    opponent = game_df["opponent"].iloc[0] if "opponent" in game_df.columns else ""
    score_str = f"{total_runs}-{opponent_runs}"

    metrics = compute_battle_metrics_table(game_df)
    row_dict = {
        "GameDate": game_date,
        "Opponent": opponent,
        "Score": score_str
    }
    for key, vals in metrics.items():
        row_dict[key] = vals["display"]
        row_dict[key + "__met"] = bool(vals["met"])
        row_dict[key + "__vs_goal_pct"] = float(vals["vs_goal_pct"]) if "vs_goal_pct" in vals and pd.notnull(vals["vs_goal_pct"]) else None
        season_summ_rows.append({
            "Metric": key,
            "Met": 1 if bool(vals["met"]) else 0,
            "VsGoalPct": float(vals["vs_goal_pct"]) if pd.notnull(vals["vs_goal_pct"]) else np.nan,
            "GameId": game_id
        })
    season_rows.append(row_dict)

# Season combined PDF with two pages
if season_rows:
    season_df = pd.DataFrame(season_rows)
    season_summary_df = (
        pd.DataFrame(season_summ_rows)
        .groupby("Metric", as_index=False)
        .agg(
            Games=("GameId", "nunique"),
            Games_Met=("Met", "sum"),
            Avg_Vs_Goal_Pct=("VsGoalPct", "mean")
        )
        .sort_values("Metric")
    )

    combined_pdf_path = os.path.join(scorecards_folder, "Season_Battle_Summary.pdf")
    render_season_summary_pdf(season_df, season_summary_df, combined_pdf_path)


In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

# Define performance goals based on the 2025 OFFENSE BATTLES document
GOALS = {
    "Battle 1: Leadoff Baserunners": {
        "Leadoff Runners (Offense)": 4,   # Offense goal: get to 4+ leadoff runners
        "Leadoff Runners (Defense)": 3    # Defense goal: 3 or fewer allowed
    },
    "Battle 2: Score the Leadoff Baserunners": {
        "Leadoff Runs % (Offense)": 67,     # Offense goal: 67% or higher
        "Leadoff Stranded % (Defense)": 70  # Defense goal: 70% or higher
    },
    "Battle 3: Total Baserunners": {
        "Total Baserunners (Offense)": 16,  # Offense goal: get to 16+ baserunners
        "Total BR + Extra Bases (Offense)": 24,
        "Total Baserunners (Defense)": 13   # Defense goal: 13 or fewer allowed
    },
    "Battle 4: Clean Defense": {
        "Defensive Errors (Pitching)": 0   # Goal: 0 USD errors
    },
    "Battle 5: Free 90s": {
        "BB + HBP (Offense)": "More than K",  # Offense goal: more walks/HBP than strikeouts (not directly calculated here)
        "BB + HBP (Defense)": 3              # Defense goal: 3 or fewer allowed
    }
}

def compute_battle_performance(df):
    """
    Computes battle metrics for USD, separating batting and pitching performances.
    Includes numerator and denominator for percentage calculations and 
    compares actual values to defined goals.
    """
    # Filter for USD's batting and pitching data
    batting_df = df[df["battingTeam"] == "USD"].copy()
    pitching_df = df[df["pitchingTeam"] == "USD"].copy()

    # Battle 1: Leadoff Baserunners
    total_innings_batting = batting_df["inn"].nunique()
    leadoff_success_bat = batting_df["inning_leadoff"].sum()  # offense
    leadoff_success_pitch = pitching_df["inning_leadoff"].sum()  # defense

    # Battle 2: Score the Leadoff Baserunners (Offense)
    if leadoff_success_bat > 0:
        grouped_inn_bat = batting_df.groupby("inn").agg({
            "inning_leadoff": "max",
            "Runs Scored": "max"
        }).reset_index()
        innings_with_runs = grouped_inn_bat[
            (grouped_inn_bat["inning_leadoff"] == 1) & (grouped_inn_bat["Runs Scored"] > 0)
        ].shape[0]
        leadoff_runs_pct = (innings_with_runs / leadoff_success_bat) * 100
        leadoff_runs_text = f"{innings_with_runs}/{leadoff_success_bat} ({leadoff_runs_pct:.1f}%)"
        leadoff_runs_met = leadoff_runs_pct >= GOALS["Battle 2: Score the Leadoff Baserunners"]["Leadoff Runs % (Offense)"]
    else:
        leadoff_runs_text, leadoff_runs_met = "0/0 (0.0%)", False

    # Battle 2: Score the Leadoff Baserunners (Defense)
    if leadoff_success_pitch > 0:
        grouped_inn_pitch = pitching_df.groupby("inn").agg({
            "inning_leadoff": "max",
            "Runs Scored": "max"
        }).reset_index()
        innings_with_runs_allowed = grouped_inn_pitch[
            (grouped_inn_pitch["inning_leadoff"] == 1) & (grouped_inn_pitch["Runs Scored"] > 0)
        ].shape[0]
        innings_leadoff_stranded = leadoff_success_pitch - innings_with_runs_allowed
        leadoff_stranded_pct = (innings_leadoff_stranded / leadoff_success_pitch) * 100
        leadoff_stranded_text = f"{innings_leadoff_stranded}/{leadoff_success_pitch} ({leadoff_stranded_pct:.1f}%)"
        leadoff_stranded_met = leadoff_stranded_pct >= GOALS["Battle 2: Score the Leadoff Baserunners"]["Leadoff Stranded % (Defense)"]
    else:
        leadoff_stranded_text, leadoff_stranded_met = "0/0 (0.0%)", False

    # Battle 3: Total Baserunners
    baserunner_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}
    total_baserunners_off = batting_df[batting_df["event_category"].isin(baserunner_events)].shape[0]
    total_baserunners_def = pitching_df[pitching_df["event_category"].isin(baserunner_events)].shape[0]
    # NEW: Total Bases + Extra Bases (any_adv)
    tb_off = int(batting_df['total_bases'].fillna(0).sum()) if 'total_bases' in batting_df.columns else 0
    xb_off = int(batting_df['any_adv'].fillna(0).sum()) if 'any_adv' in batting_df.columns else 0
    tb_plus_xb_off = tb_off + xb_off

    # Battle 4: Clean Defense
    defensive_errors = pitching_df["pitchResult"].str.contains("error", case=False, na=False).sum()

    # Battle 5: Free 90s
    free_90s_off = batting_df[batting_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0]
    strikeouts_off = batting_df['pitchResult'].str.contains("Strikeout", na=False).sum()
    free_90s_def = pitching_df[pitching_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0]

    return {
        "Battle 1: Leadoff Baserunners": {
            "Leadoff Runners (Offense)": f"{leadoff_success_bat} (Goal: 4+) {'✔' if leadoff_success_bat >= 4 else '✘'}",
            "Leadoff Runners (Defense)": f"{leadoff_success_pitch} (Goal: ≤3) {'✔' if leadoff_success_pitch <= 3 else '✘'}"
        },
        "Battle 2: Score the Leadoff Baserunners": {
            "Leadoff Runs % (Offense)": f"{leadoff_runs_text} (Goal: 67%+) {'✔' if leadoff_runs_met else '✘'}",
            "Leadoff Stranded % (Defense)": f"{leadoff_stranded_text} (Goal: 70%+) {'✔' if leadoff_stranded_met else '✘'}"
        },
        "Battle 3: Total Baserunners": {
            "Total Baserunners (Offense)": f"{total_baserunners_off} (Goal: 16+) {'✔' if total_baserunners_off >= 16 else '✘'}",
            "Total BR + Extra Bases (Offense)": f"{tb_plus_xb_off} (Goal: 24+) {'✔' if tb_plus_xb_off >= 24 else '✘'}",
            "Total Baserunners (Defense)": f"{total_baserunners_def} (Goal: ≤13) {'✔' if total_baserunners_def <= 13 else '✘'}"
        },
        "Battle 4: Clean Defense": {
            "Defensive Errors (Pitching)": f"{defensive_errors} (Goal: 0) {'✔' if defensive_errors == 0 else '✘'}"
        },
        "Battle 5: Free 90s": {
            "BB + HBP (Offense)": f"{free_90s_off} (Goal: More than K ({strikeouts_off})) {'✔' if free_90s_off > strikeouts_off else '✘'}",
            "BB + HBP (Defense)": f"{free_90s_def} (Goal: ≤3) {'✔' if free_90s_def <= 3 else '✘'}"
        }
    }

def create_battle_report_text(df):
    """
    Computes the battle performance report and returns it as a formatted string.
    """
    battle_results = compute_battle_performance(df)
    game_date = df['gameDate'].iloc[0].strftime('%Y-%m-%d')
    
    report_lines = [f"=== USD Battle Performance Report (Battles 1–5) - {game_date} ===\n"]
    for battle_name, metrics in battle_results.items():
        report_lines.append(battle_name)
        for metric_label, metric_value in metrics.items():
            report_lines.append(f"   {metric_label}: {metric_value}")
        report_lines.append("")
    return "\n".join(report_lines)

# --- NEW: helpers for combined PDF table (season-long) ---

def compute_battle_metrics_table(df):
    """
    Returns per-metric display strings like '3/4' (or similar) and met flags for coloring,
    plus components needed to compute season summaries.
    Output dict keys are canonical metric codes matching table columns.
    """
    batting_df = df[df["battingTeam"] == "USD"].copy()
    pitching_df = df[df["pitchingTeam"] == "USD"].copy()

    out = {}

    # Battle 1: Leadoff Runners
    leadoff_off = int(batting_df["inning_leadoff"].sum())
    leadoff_def = int(pitching_df["inning_leadoff"].sum())
    goal_1a = 4
    goal_1b = 3
    out["B1a Leadoff Runners (Off)"] = {
        "display": f"{leadoff_off}/{goal_1a}",
        "met": leadoff_off >= goal_1a,
        "value": leadoff_off,
        "goal": goal_1a,
        "vs_goal_pct": (leadoff_off / goal_1a * 100.0) if goal_1a else np.nan
    }
    out["B1b Leadoff Runners (Def)"] = {
        "display": f"{leadoff_def}/{goal_1b}",
        "met": leadoff_def <= goal_1b,
        "value": leadoff_def,
        "goal": goal_1b,
        "vs_goal_pct": (goal_1b / leadoff_def * 100.0) if leadoff_def > 0 else 100.0
    }

    # Battle 2: Leadoff scored / stranded
    if leadoff_off > 0:
        grp_b = batting_df.groupby("inn").agg({"inning_leadoff": "max", "Runs Scored": "max"}).reset_index()
        innings_with_runs = grp_b[(grp_b["inning_leadoff"] == 1) & (grp_b["Runs Scored"] > 0)].shape[0]
        pct_off = (innings_with_runs / leadoff_off) * 100.0
        display_off = f"{innings_with_runs}/{leadoff_off}"
    else:
        innings_with_runs, pct_off, display_off = 0, 0.0, "0/0"

    if leadoff_def > 0:
        grp_p = pitching_df.groupby("inn").agg({"inning_leadoff": "max", "Runs Scored": "max"}).reset_index()
        runs_allowed = grp_p[(grp_p["inning_leadoff"] == 1) & (grp_p["Runs Scored"] > 0)].shape[0]
        stranded = leadoff_def - runs_allowed
        pct_def = (stranded / leadoff_def) * 100.0
        display_def = f"{stranded}/{leadoff_def}"
    else:
        stranded, pct_def, display_def = 0, 0.0, "0/0"

    out["B2a Leadoff Runs % (Off)"] = {
        "display": display_off,  # numerator/denominator
        "met": pct_off >= 67.0,
        "value": pct_off,
        "goal": 67.0,
        "vs_goal_pct": (pct_off / 67.0 * 100.0) if 67.0 else np.nan
    }
    out["B2b Leadoff Stranded % (Def)"] = {
        "display": display_def,  # stranded/total leadoff reached
        "met": pct_def >= 70.0,
        "value": pct_def,
        "goal": 70.0,
        "vs_goal_pct": (pct_def / 70.0 * 100.0) if 70.0 else np.nan
    }

    # Battle 3: Total Baserunners
    baserunner_events = {"single", "double", "triple", "home_run", "walk", "hit_by_pitch"}
    tot_off = int(batting_df[batting_df["event_category"].isin(baserunner_events)].shape[0])
    tot_def = int(pitching_df[pitching_df["event_category"].isin(baserunner_events)].shape[0])
    goal_3a, goal_3b = 16, 13
    out["B3a Total Baserunners (Off)"] = {
        "display": f"{tot_off}/{goal_3a}",
        "met": tot_off >= goal_3a,
        "value": tot_off,
        "goal": goal_3a,
        "vs_goal_pct": (tot_off / goal_3a * 100.0)
    }
    out["B3b Total Baserunners (Def)"] = {
        "display": f"{tot_def}/{goal_3b}",
        "met": tot_def <= goal_3b,
        "value": tot_def,
        "goal": goal_3b,
        "vs_goal_pct": (goal_3b / tot_def * 100.0) if tot_def > 0 else 100.0
    }
    
    tb_off = int(batting_df['total_bases'].fillna(0).sum()) if 'total_bases' in batting_df.columns else 0
    xb_off = int(batting_df['any_adv'].fillna(0).sum()) if 'any_adv' in batting_df.columns else 0
    tb_plus_xb_off = tb_off + xb_off

    out["B3c Total Bases + XBs (Off)"] = {
        "display": f"{tb_plus_xb_off}/24",
        "met": tb_plus_xb_off >= 24,
        "value": tb_plus_xb_off,
        "goal": 24,
        "vs_goal_pct": (tb_plus_xb_off / 24 * 100.0)
    }


    # Battle 4: Errors
    errors = int(pitching_df["pitchResult"].str.contains("error", case=False, na=False).sum())
    out["B4 Defensive Errors (Pitch)"] = {
        "display": f"{errors}/0",
        "met": errors == 0,
        "value": errors,
        "goal": 0.0,
        "vs_goal_pct": 100.0 if errors == 0 else 0.0
    }

    # Battle 5: Free 90s
    free90_off = int(batting_df[batting_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0])
    ks_off = int(batting_df['pitchResult'].str.contains("Strikeout", na=False).sum())
    free90_def = int(pitching_df[pitching_df["event_category"].isin(["walk", "hit_by_pitch"])].shape[0])

    out["B5a BB+HBP (Off) vs K"] = {
        "display": f"{free90_off}/{ks_off}",
        "met": free90_off > ks_off,
        "value": free90_off,
        "goal": float(ks_off),
        "vs_goal_pct": (free90_off / max(ks_off, 1) * 100.0)
    }
    out["B5b BB+HBP (Def)"] = {
        "display": f"{free90_def}/3",
        "met": free90_def <= 3,
        "value": free90_def,
        "goal": 3.0,
        "vs_goal_pct": (3.0 / free90_def * 100.0) if free90_def > 0 else 100.0
    }

    return out

def wrap_col_header(header, width=16):
    # Wrap header string to a given width for table display
    return "\n".join(textwrap.wrap(header, width=width, break_long_words=False, replace_whitespace=False))

def render_season_summary_pdf(rows_df, summary_df, out_path):
    """
    Render a season-long combined PDF with:
      - Page 1: A table: rows=games, columns=battle metrics, cells like '3/4' and green/red by met.
        Column headers are wrapped and column widths are natural.
      - Page 2: A totals summary table with Games Met and Avg % of Goal.
    """
    metric_cols = [
        "B1a Leadoff Runners (Off)",
        "B1b Leadoff Runners (Def)",
        "B2a Leadoff Runs % (Off)",
        "B2b Leadoff Stranded % (Def)",
        "B3a Total Baserunners (Off)",
        "B3c Total Bases + XBs (Off)",
        "B3b Total Baserunners (Def)",
        "B4 Defensive Errors (Pitch)",
        "B5a BB+HBP (Off) vs K",
        "B5b BB+HBP (Def)",
    ]
    display_cols = ["GameDate", "GameId"] + metric_cols

    # Wrap column headers for display
    wrapped_headers = [wrap_col_header(col, width=18) for col in display_cols]

    # Build display table (strings) and color flags
    disp = rows_df[display_cols].copy()

    # Calculate natural column widths based on header and cell content
    def get_col_width(col, header, min_width=0.08, max_width=0.22):
        # Estimate width: max of header and cell content (in characters)
        max_len = max([len(str(header))] + [len(str(x)) for x in disp[col]])
        # Scale: 0.08 for short cols, up to 0.22 for long
        if col in ["GameDate", "GameId"]:
            return 0.10
        elif max_len <= 8:
            return 0.10
        elif max_len <= 14:
            return 0.13
        elif max_len <= 20:
            return 0.16
        else:
            return min(max_width, 0.10 + 0.01 * max_len)
    col_widths = [get_col_width(col, header) for col, header in zip(display_cols, wrapped_headers)]

    # Create cell colors: default white, then set by met flags
    cell_colours = [["white"] * len(display_cols) for _ in range(len(disp))]
    for r_idx, row in rows_df.iterrows():
        for c_idx, col in enumerate(display_cols):
            if col in ["GameDate", "GameId"]:
                cell_colours[r_idx][c_idx] = "white"
            else:
                cell_colours[r_idx][c_idx] = "#c6efce" if row.get(col + "__met", False) else "#ffc7ce"

    # Build table data
    table_data = [wrapped_headers] + disp.values.tolist()
    colours = [["#d9d9d9"] * len(display_cols)] + cell_colours  # header grey

    table = ax_table.table(cellText=table_data,
                           cellLoc='center',
                           colLabels=None,
                           colColours=None,
                           cellColours=colours,
                           loc='upper left')

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)

    # Bold header row
    for c in range(len(display_cols)):
        cell = table[0, c]
        cell.set_text_props(fontweight='bold')

    # Column widths (tune for readability)
    col_widths = []
    for c, col in enumerate(display_cols):
        if col in ["GameDate", "GameId"]:
            col_widths.append(0.08 if col == "GameDate" else 0.08)
        else:
            col_widths.append(0.11)
    for c, w in enumerate(col_widths):
        table.auto_set_column_width(col=list(range(len(display_cols))))
        # matplotlib's table doesn't support per-column width directly via API across backends
        # but auto_set_column_width helps. We'll keep as is for portability.

    # Footer summary table
    ax_sum = plt.axes([0.02, 0.05, 0.96, 0.14])
    ax_sum.axis("off")

    # Build summary display
    sum_headers = ["Metric", "Games Met", "Avg % of Goal"]
    sum_rows = []
    for metric in metric_cols:
        row = summary_df.loc[summary_df["Metric"] == metric]
        if not row.empty:
            games = int(row["Games"].values[0])
            met = int(row["Games_Met"].values[0])
            avg_pct = row["Avg_Vs_Goal_Pct"].values[0]
            sum_rows.append([
                metric,
                f"{met}/{games}",
                f"{avg_pct:.1f}%"
            ])
        else:
            sum_rows.append([metric, "0/0", "—"])

    sum_table_data = [sum_headers] + sum_rows
    sum_colours = [["#d9d9d9", "#d9d9d9", "#d9d9d9"]] + [["white", "white", "white"] for _ in sum_rows]

    sum_table = ax_sum.table(cellText=sum_table_data,
                             cellLoc='center',
                             cellColours=sum_colours,
                             loc='upper left')
    sum_table.auto_set_font_size(False)
    sum_table.set_fontsize(10)
    sum_table.scale(1, 1.5)

    for c in range(len(sum_headers)):
        cell = sum_table[0, c]
        cell.set_text_props(fontweight='bold')

    # Save
    fig.savefig(out_path, format="pdf", bbox_inches="tight", facecolor='white')
    plt.close(fig)

# Loop through each unique gameId and generate a PDF report.
# PDFs are saved to the scorecards folder.
scorecards_folder = r"C:\Users\TrevorWhite\OneDrive - Good360\Documents\bsb\scorecards"
os.makedirs(scorecards_folder, exist_ok=True)

# Collect per-game numeric & display summaries for combined season PDF
season_rows = []
season_summ_rows = []

for game_id in df_joined["gameId"].unique():
    game_df = df_joined[df_joined["gameId"] == game_id]
    if game_df.empty:
        continue
    game_date = game_df['gameDate'].iloc[0].strftime('%Y-%m-%d')
    report_text = create_battle_report_text(game_df)
    
    fig, ax = plt.subplots(figsize=(8.5, 11), facecolor='white')
    ax.set_facecolor('white')
    ax.axis("off")
    ax.text(0.05, 0.95, report_text, transform=ax.transAxes, fontsize=14, va="top", family="monospace", color="black")
    
    file_name = f"BattleReport_{game_date}_{game_id}.pdf"
    save_path = scorecards_folder + "\\" + file_name
    fig.savefig(save_path, format="pdf", bbox_inches="tight", facecolor='white')
    plt.close(fig)

    # --- accumulate rows for season table ---
    metrics = compute_battle_metrics_table(game_df)
    row_dict = {"GameDate": game_date, "GameId": str(game_id)}
    for key, vals in metrics.items():
        row_dict[key] = vals["display"]
        row_dict[key + "__met"] = bool(vals["met"])
        season_summ_rows.append({
            "Metric": key,
            "Met": 1 if bool(vals["met"]) else 0,
            "VsGoalPct": float(vals["vs_goal_pct"]) if pd.notnull(vals["vs_goal_pct"]) else np.nan,
            "GameId": game_id
        })
    season_rows.append(row_dict)

# Build and save combined season PDF (table with per-game rows + totals summary)
if season_rows:
    season_df = pd.DataFrame(season_rows).sort_values("GameDate", ascending=True)


    # Summary aggregation
    season_summary_df = (
        pd.DataFrame(season_summ_rows)
        .groupby("Metric", as_index=False)
        .agg(
            Games=("GameId", "nunique"),
            Games_Met=("Met", "sum"),
            Avg_Vs_Goal_Pct=("VsGoalPct", "mean")
        )
        .sort_values("Metric")
    )

    combined_pdf_path = os.path.join(scorecards_folder, "Season_Battle_Summary.pdf")
    render_season_summary_pdf(season_df, season_summary_df, combined_pdf_path)

# (Per-game PDFs unchanged; combined season PDF added.)


In [ ]:

# Sorting df_joined by gameDate in descending order and selecting specific columns
df_joined[["gameDate", "battingTeam", "inn", "pitchResult","abNumInGame", "event_category", 
                        "inning_leadoff", "inning_leadoff_success", "Runs Scored"]] \
                        .sort_values(by="gameDate", ascending=True) \
                        .head(500)


In [ ]:
# ... existing code ...

# Add these pandas display options at the beginning of your notebook, after the imports
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Width of the display in characters
pd.set_option('display.max_colwidth', None)  # Show full content of each column

# ... existing code ...
# df_transformed.head(250)

In [ ]:
df_joined.head()

In [ ]:
import pandas as pd

# Load the dataset
file_path = "C:/Users/TrevorWhite/Downloads/NCAA_STUFF_PLUS_24_TRAIN.csv"
df = pd.read_csv(file_path)

# Optionally, normalize column names (remove extra spaces and lower-case)
df.columns = df.columns.str.strip().str.lower()

# Convert gameDate to datetime and extract year (adjust the column name if necessary)
df['gamedate'] = pd.to_datetime(df['gamedate'], errors='coerce')
df['year'] = df['gamedate'].dt.year

# Treat FA and FF as the same pitch type (convert them to 'FB')
df['pitch_type'] = df['pitch_type'].replace({'FA': 'FB', 'FF': 'FB'})

# Define the columns to average
cols_to_avg = [
    'spin_rate', 'extension', 'horzapprangle', 'vertapprangle',
    'horzrelangle', 'vertrelangle', 'az', 'ax', 'z0', 'x0', 'start_speed'
]

# Convert these columns to numeric to ensure proper aggregation
for col in cols_to_avg:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Optionally drop rows where any of these numeric columns is missing
df = df.dropna(subset=cols_to_avg)

# Group by pitcherid, pitch_type, and year, computing means and count.
# (Note: we assume the pitcher id column is called "pitcherid" after normalization.
#  If it's "pitcherid" in your data, this is fine; if not, adjust accordingly.)
df_summary = df.groupby(['pitcherid', 'pitch_type', 'year'], as_index=False).agg(
    **{col: (col, 'mean') for col in cols_to_avg},
    pitch_count=('pitch_type', 'count')
)

# Calculate total pitches per pitcher per year
df_total_pitches = df.groupby(['pitcherid', 'year'], as_index=False)['pitch_type'].count()\
                    .rename(columns={'pitch_type': 'total_pitches'})

# Merge to compute pitch usage percentage
df_summary = df_summary.merge(df_total_pitches, on=['pitcherid', 'year'])
df_summary['usage_pct'] = df_summary['pitch_count'] / df_summary['total_pitches']

# Verify that pitch_type is present
print("df_summary columns:", df_summary.columns)

# Identify primary fastball (FB) or fallback to SI if FB is missing.
# For each pitcherid and year, we want to use FB if available; otherwise, use SI.
fastball_df = df_summary[df_summary['pitch_type'] == 'FB']
si_fallback = df_summary[df_summary['pitch_type'] == 'SI']

# Combine FB and SI rows, dropping duplicates by pitcherid and year (FB gets priority)
fastball_final = pd.concat([fastball_df, si_fallback]).drop_duplicates(subset=['pitcherid', 'year'], keep='first')

# Rename columns in the fastball summary for merging (append _FB)
fastball_final = fastball_final[['pitcherid', 'year'] + cols_to_avg]\
                    .rename(columns={col: f"{col}_FB" for col in cols_to_avg})

# Merge Fastball statistics to compute differences
df_summary = df_summary.merge(fastball_final, on=['pitcherid', 'year'], how='left')

# Compute differences (excluding spin_rate if not needed)
for col in cols_to_avg[1:]:
    df_summary[f"{col}_diff"] = df_summary[col] - df_summary[f"{col}_FB"]

# Rank metrics within each pitch_type and year group
for col in cols_to_avg:
    df_summary[f"{col}_rank"] = df_summary.groupby(['pitch_type', 'year'])[col].rank(method='dense', ascending=False)

# Save the processed data
df_summary.head()

In [ ]:
df.head()